# Dissertation figures — one notebook, organised by research question

Every figure in the dissertation is built here, from the CSVs the evaluation
array writes. This notebook replaces seven overlapping plotting modules
(`plot_results`, `plot_detail`, `plot_dissertation`, `plot_rq3_v2`,
`plot_step_curves`, `plot_loss_curves`, `plot_listening`) and the figure half
of `listening-tests/analysis/analyse_listening.py`.

**What changed in the consolidation** — three conflicts between those modules
were resolved rather than carried over:

1. **RQ3 is on the corrected scale only.** The old headline figure put a
   distance *ratio* (`seg_closure`) and a signed scalar *projection*
   (`supra_closure_mean`) on one axis as if 1.0 meant the same in both, and the
   composite was an unbounded mean of ratios that a single near-degenerate
   denominator could dominate (Dutch `f0_mean_closure` = 27.7). §5 uses two
   distances to the natural target, each over its own α=0 baseline. §8.4 keeps
   the old composite *once*, as the diagnostic that justifies retiring it.
2. **The listening figures use the fielded data**, not the pre-registration
   simulator. The old module simulated ratings, hard-coded a table of AccentCS
   values, and assumed an *accentedness* scale with a rated natural anchor. The
   test as fielded rated *similarity to a reference accent*, block A / L1 prompt
   only, with the anchor played but not rated. §6 reads the real ratings and
   recomputes AccentCS from the result CSVs.
3. **One copy of each helper.** `_save`, `recover_natural` (two versions that had
   drifted apart), `noise_sd` and the pooling helpers each existed 2–5 times.

**Layout**

| § | Research question | Chapter |
|---|---|---|
| 1 | Setup — style, loading, shared helpers | — |
| 2 | RQ1 — does α buy accent, and what does it cost? | Ch.2 (L1 prompt) |
| 3 | RQ2a — accent conversion: recovery under the GAE prompt | Ch.3 |
| 4 | RQ2b — trajectory: when does each metric stop moving? | Ch.4 |
| 5 | RQ3 — segmental vs prosodic transfer (corrected scale) | Ch.4 |
| 6 | Listening test — subjective validation | Ch.2 |
| 7 | Fine-tuning loss curves (asset A0) | Ch.2 |
| 8 | Diagnostics and appendix figures | appendix |

Run top to bottom: §1 loads every accent tree once into `DATA`, and each later
section is independent of the others.

## 1. Setup

### 1.1 Configuration

In [ ]:
from pathlib import Path

# Locate the AccentVector directory (holds results/ and accent_vector/) from
# wherever the kernel happens to have started.
ROOT = Path.cwd().resolve()
while not ((ROOT / "results").is_dir() and (ROOT / "accent_vector").is_dir()):
    if ROOT == ROOT.parent:
        raise SystemExit("run this notebook from inside the AccentVector directory")
    ROOT = ROOT.parent

REPO = ROOT.parent
RESULTS = ROOT / "results"
FIGDIR = RESULTS / "figures"
TAG = "lr3e5_r16"
ACCENTS = ["dutch", "bengali", "arabic", "hindi", "mandarin"]

# Fielded listening test (block A / L1 prompt), written by
# listening-tests/analysis/analyse_listening.py from the Qualtrics exports.
LISTENING_CSV = REPO / "listening-tests" / "analysis" / "out" / "listening_long.csv"
# TensorBoard event files from the fine-tunes; rsync down from Eddie (gitignored).
RUNS_ROOT = ROOT / "runs"

SAVE = True          # write .pdf + .png beside each figure as well as showing it

print(f"root      {ROOT}")
print(f"results   {RESULTS}  (tag {TAG})")
print(f"figures   {FIGDIR}")

### 1.2 House style

Okabe–Ito, colour is never the only cue (every series also carries a marker and
a line style), recessive grid and axes, one legend per figure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

%matplotlib inline

INK, MUTED, GRID = "#222222", "#666666", "#cccccc"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "legend.frameon": False, "figure.autolayout": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
    # save() leaves figures open so the inline backend can display them;
    # Jupyter closes them after each cell, a headless run does not.
    "figure.max_open_warning": 0,
})

# One entry per accent: colour + a redundant marker / line-style cue.
ACCENT_STYLE = {
    "dutch":    {"c": "#0072B2", "ls": "-",               "m": "o", "label": "Dutch"},
    "bengali":  {"c": "#D55E00", "ls": "--",              "m": "s", "label": "Bengali"},
    "arabic":   {"c": "#009E73", "ls": "-.",              "m": "^", "label": "Arabic"},
    "hindi":    {"c": "#CC79A7", "ls": ":",               "m": "D", "label": "Hindi"},
    "british":  {"c": "#56B4E9", "ls": (0, (3, 1, 1, 1)), "m": "v", "label": "British"},
    "mandarin": {"c": "#E69F00", "ls": (0, (5, 1)),       "m": "P", "label": "Mandarin"},
}
SPEAKER_STYLE = {"m": {"ls": "--", "m": "v", "label": "male prompt"},
                 "f": {"ls": ":",  "m": "^", "label": "female prompt"}}
# Lines for the α series on the Ch.4 slope panels: styled, not a colour ramp.
ALPHA_STYLE = [("#0072B2", "-", "o"), ("#D55E00", "--", "s"), ("#009E73", "-.", "^"),
               ("#CC79A7", ":", "D"), ("#E69F00", (0, (3, 1, 1, 1)), "v")]

REF_STYLE = {"l1":     {"c": "#0072B2", "ls": "-",  "m": "o"},
             "native": {"c": "#D55E00", "ls": "--", "m": "s"}}
REF_LABEL = {"l1": "L1 prompt", "native": "GAE prompt"}
REF_SLUG = {"l1": "l1", "native": "gae"}

# Sequential (unsigned magnitudes) and diverging (signed closure) ramps.
SEQ = LinearSegmentedColormap.from_list(
    "seqblue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("divbr", ["#d03b3b", "#f0efec", "#2a78d6"])

SEG_C, SUP_C = "#0072B2", "#D55E00"      # RQ3 channels: segmental / prosodic


def style(accent):
    return ACCENT_STYLE.get(accent, {"c": INK, "ls": "-", "m": "o", "label": accent})


def save(fig, name, subdir="dissertation"):
    """Write one figure as .pdf + .png under results/figures/<subdir>/.

    Deliberately does NOT close the figure: with the inline backend, closing here
    would suppress the display the notebook exists for.
    """
    if not SAVE:
        return
    out = FIGDIR / subdir / name
    out.parent.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(out.with_suffix(f".{ext}"), bbox_inches="tight")
    print(f"wrote {out.relative_to(ROOT)}.pdf (+ .png)")

### 1.3 Loading the metric tree

The eval array writes
`results/<accent>/<tag>/<ref_kind>/<speaker>/metrics/step_<N>/{rq1,rq3,utmos}.csv`.
`load_tree` flattens one accent into a long frame (one row per
ref_kind × speaker × step × α) plus a frame of the `# key=val` footer summaries
that `rq1.csv` carries.

In [ ]:
import re


def _read_footer(path):
    """Parse the trailing '# key=val, ...' summary line of an rq CSV into a dict."""
    out, last = {}, ""
    for line in Path(path).read_text().splitlines():
        if line.startswith("#"):
            last = line
    for m in re.finditer(r"(\w+)=([-\d.eE]+|nan)", last):
        try:
            out[m.group(1)] = float(m.group(2))
        except ValueError:
            pass
    return out


def _read_csv(path):
    return pd.read_csv(path, comment="#") if Path(path).exists() else None


def load_tree(root):
    """(long_df, footer_df) for one results/<accent>/<tag> tree."""
    root = Path(root)
    rows, footers = [], []
    for ref in ("l1", "native"):
        for spk in ("m", "f"):
            mdir = root / ref / spk / "metrics"
            if not mdir.is_dir():
                continue
            for step_dir in sorted(mdir.glob("step_*")):
                step = int(step_dir.name.split("_")[1])
                rq1 = _read_csv(step_dir / "rq1.csv")
                if rq1 is None:
                    continue
                df = rq1.copy()
                for extra in ("rq3.csv", "utmos.csv"):
                    e = _read_csv(step_dir / extra)
                    if e is not None:
                        df = df.merge(e, on="alpha", how="left", suffixes=("", "_dup"))
                df["ref_kind"], df["speaker"], df["step"] = ref, spk, step
                rows.append(df)
                f = _read_footer(step_dir / "rq1.csv")
                f.update(ref_kind=ref, speaker=spk, step=step)
                footers.append(f)
    if not rows:
        raise FileNotFoundError(f"no rq1.csv under {root}/<ref>/<spk>/metrics/step_*/")
    return pd.concat(rows, ignore_index=True), pd.DataFrame(footers)


def load_accents(results_root=RESULTS, tag=TAG, accents=ACCENTS):
    """{accent: long_df}, {accent: footer_df} for every accent tree that loads."""
    data, foot = {}, {}
    for acc in accents:
        root = Path(results_root) / acc / tag
        try:
            long_df, footer = load_tree(root)
        except FileNotFoundError as e:
            print(f"skip {acc}: {e}")
            continue
        data[acc], foot[acc] = long_df, footer
        print(f"{acc:>9}: refs={sorted(long_df.ref_kind.unique())} "
              f"steps={len(long_df.step.unique())} "
              f"(to {long_df.step.max():,}) α={len(long_df.alpha.unique())}")
    return data, foot


def pool_alpha(df, col):
    """Speaker-pooled mean + min/max band over α, for one (ref, step) slice."""
    d = df.dropna(subset=[col]) if col in df.columns else df.iloc[:0]
    if d.empty:
        return pd.DataFrame(columns=["alpha", "mean", "lo", "hi"])
    return (d.groupby("alpha")[col].agg(mean="mean", lo="min", hi="max")
             .reset_index().sort_values("alpha"))


def pooled(df, col, ref, step):
    """pool_alpha for one (ref_kind, step) of a long frame."""
    return pool_alpha(df[(df.ref_kind == ref) & (df.step == step)], col)

### 1.4 The metric registry

One place naming every plotted column, its axis label and its better-direction.
Adding a metric to the CSVs means adding one row here.

In [ ]:
# (column, panel title, y-axis label, better direction, scale factor)
PANELS = [
    ("wer",                   "Word error rate",           "WER (%)",        "↓", 100.0),
    ("accent_cs",             "Accent similarity",         "cosine sim.",    "↑", 1.0),
    ("eng_lid",               "Language ID",               "P(English)",     "↑", 1.0),
    ("seg_ppg_kl_to_natural", "Phonetic posteriorgram KL", "sym. KL (nats)", "↓", 1.0),
    ("spk_sim",               "Speaker similarity",        "cosine sim.",    "↑", 1.0),
    ("utmos",                 "UTMOS",                     "predicted MOS",  "↑", 1.0),
]

# The cross-accent overview drops KL-PPG and UTMOS: neither separates the accents.
# KL-PPG is dominated by a per-accent constant (the ground-truth speaker is not the
# prompt speaker for four of five accents), so its lines are flat offsets; UTMOS
# carries an unquantified bias against non-native accents. Both stay on the
# per-accent figures, where the α-trend is what is read.
OVERVIEW_DROP = ("seg_ppg_kl_to_natural", "utmos")
OVERVIEW_PANELS = [p for p in PANELS if p[0] not in OVERVIEW_DROP]

# The six suprasegmental descriptors rq3.csv stores, in their own units.
SUPRA = ["pct_voiced", "npvi_voiced", "artic_rate", "f0_mean", "f0_std", "f0_range"]
SUPRA_LABEL = {"pct_voiced": "Voiced fraction (%V proxy)", "npvi_voiced": "nPVI over voiced runs",
               "artic_rate": "Articulation rate", "f0_mean": "F0 mean",
               "f0_std": "F0 std", "f0_range": "F0 range"}
SUPRA_UNIT = {"pct_voiced": "fraction of frames", "npvi_voiced": "nPVI",
              "artic_rate": "voiced runs / s", "f0_mean": "Hz", "f0_std": "Hz",
              "f0_range": "Hz"}

WER_UNUSABLE = 0.20     # above this the output is not usable English any more

### 1.5 Shared statistics

Both the Ch.4 trajectory section and the RQ3 error bars need an estimate of
evaluation noise. There is no within-condition SD in the CSVs (they store
per-α means over the n utterances), so noise is estimated from the
checkpoint-to-checkpoint jitter of the metric itself.

In [ ]:
def noise_sd(values):
    """Robust SD of checkpoint-to-checkpoint jitter (successive-difference estimator)."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size < 4:
        return np.nan
    d = np.diff(v)
    return float(1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2))


def rolling_slope(steps, values, window=5):
    """Local OLS slope (metric units per 1,000 steps) in a centred window.

    Point-to-point differences are dominated by evaluation noise (5 utterances per
    speaker per checkpoint), so each point is the slope of a least-squares line
    through the `window` checkpoints around it, not a raw finite difference.
    """
    steps = np.asarray(steps, dtype=float)
    values = np.asarray(values, dtype=float)
    half = window // 2
    out = np.full(len(steps), np.nan)
    for i in range(len(steps)):
        lo, hi = max(0, i - half), min(len(steps), i + half + 1)
        if hi - lo >= 3:
            out[i] = np.polyfit(steps[lo:hi], values[lo:hi], 1)[0] * 1000.0
    return steps, out


def settling_step(steps, values, k=2.0, min_tail=3):
    """Earliest step from which the metric stays within the evaluation-noise band.

    The band is ±k·σ around the final value, σ estimated by `noise_sd` — so
    "settled" means later training moves the metric no more than the eval noise
    does, rather than by some arbitrary fixed fraction. `min_tail` checkpoints must
    remain, so the final point can never trivially settle.

    Returns (step, band, band_frac_of_range); step is None if it never settles.
    """
    steps = np.asarray(steps, dtype=float)
    values = np.asarray(values, dtype=float)
    sigma = noise_sd(values)
    rng = np.nanmax(values) - np.nanmin(values)
    if not np.isfinite(sigma) or sigma <= 0:
        return None, np.nan, np.nan
    band = k * sigma
    frac = band / rng if rng > 0 else np.nan
    ok = np.abs(values - values[-1]) <= band
    for i in range(len(ok) - min_tail + 1):
        if ok[i:].all():
            return steps[i], band, frac
    return None, band, frac


def leak_alpha(df, ref, step, threshold=WER_UNUSABLE):
    """Lowest α whose pooled WER exceeds `threshold` (nan if it never does)."""
    g = df[(df.ref_kind == ref) & (df.step == step)]
    if "wer" not in g.columns or g.empty:
        return np.nan
    p = g.groupby("alpha")["wer"].mean().sort_index()
    over = p[p > threshold]
    return float(over.index[0]) if len(over) else np.nan


def ratio_axis(ax, which="y", ticks=(0.5, 1, 2, 4), lim=None):
    """Log axis in ratio units with clean labels — matplotlib's default log minor
    ticks render as '6 x 10^-1' and collide with the decade labels."""
    for w in which:
        axis = ax.yaxis if w == "y" else ax.xaxis
        (ax.set_yscale if w == "y" else ax.set_xscale)("log")
        (ax.set_yticks if w == "y" else ax.set_xticks)(list(ticks))
        (ax.set_yticklabels if w == "y" else ax.set_xticklabels)([f"{t:g}" for t in ticks])
        axis.set_minor_locator(plt.NullLocator())
        axis.set_minor_formatter(plt.NullFormatter())
        if lim:
            (ax.set_ylim if w == "y" else ax.set_xlim)(*lim)


def panel_axes(nrows=2, ncols=3, w=3.5, h=3.0):
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows))
    return fig, np.atleast_1d(axes).ravel()


def axes_for(panels, w=3.5, h=3.0):
    """Grid shaped to the panel count: 2 columns up to four panels, else 3."""
    n = len(panels)
    ncols = 2 if n <= 4 else 3
    return panel_axes(-(-n // ncols), ncols, w, h)


def finish_panel(ax, title, ylabel, arrow):
    ax.set_title(f"{title}  {arrow}", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.margins(x=0.04)


def legend_rows(fig, handles, per_row=3, y0=-0.035, dy=0.042, fontsize=9):
    """Centred multi-row figure legend.

    matplotlib packs a single `ncol` legend column-major, so a final row with fewer
    entries than columns hangs off to the left. One centred legend per row keeps
    every row centred under the panels.
    """
    rows = [handles[i:i + per_row] for i in range(0, len(handles), per_row)]
    for i, row in enumerate(rows):
        fig.add_artist(fig.legend(handles=row, loc="lower center", ncol=len(row),
                                  fontsize=fontsize, bbox_to_anchor=(0.5, y0 - dy * i)))

### 1.6 Load every accent once

`DATA[accent]` is the long frame, `FOOT[accent]` the footer summaries and
`FINALS[accent]` the last checkpoint that accent reached.

In [ ]:
DATA, FOOT = load_accents()
if not DATA:
    raise SystemExit("no accent trees loaded — check RESULTS and TAG")
FINALS = {a: int(df.step.max()) for a, df in DATA.items()}
print("\nfinal checkpoints:", {a: f"{s:,}" for a, s in FINALS.items()})

## 2. RQ1 — does α buy accent, and what does it cost?

Chapter 2. The L1 prompt (`ref_kind = l1`): each speaker is cloned from a
reference clip in their own L1, and α scales the accent vector on top.

The two figure functions below are the workhorses of Chapters 2 and 3 — the
same layouts are re-used in §3 with `ref="native"`, which is why they take
`ref` as an argument rather than existing twice.

### 2.1 Per-accent metric grid

One figure per accent at its final checkpoint: six metrics against α, each
prompt speaker drawn faintly beneath the pooled mean so pooling never hides a
disagreement between them.

In [ ]:
# The L1-prompt ceiling drawn onto the GAE figures in §3. RQ2 reads accent gain
# under the GAE prompt as a fraction of the interval between the GAE-prompt floor
# (α=0 on the same panel) and this ceiling; without it the accent-similarity panel
# gives no sense of scale — a rise of 0.05 looks the same whether the interval is
# 0.03 or 0.5 wide.
CEILING_PANEL = "accent_cs"


def ceiling(df, step, speaker=None):
    """Speaker-pooled AccentCS of the L1 prompt at α=0: the RQ2 ceiling."""
    d = df[(df.ref_kind == "l1") & (df.step == step) & (df.alpha == 0.0)]
    if speaker is not None:
        d = d[d.speaker == speaker]
    d = d.dropna(subset=["accent_cs"])
    return float(d.accent_cs.mean()) if not d.empty else None


def fig_alpha_single_accent(long_df, accent, ref, step=None):
    """2x3 metric grid vs α for one accent at one checkpoint."""
    d = long_df[long_df.ref_kind == ref]
    if d.empty:
        print(f"{accent}/{ref}: no rows; skipping")
        return None, None
    step = int(d.step.max()) if step is None else int(step)
    d = d[d.step == step]
    st = style(accent)

    fig, axes = panel_axes()
    drew_ceiling = False
    for ax, (col, title, ylab, arrow, scale) in zip(axes, PANELS):
        if col not in d.columns or d[col].notna().sum() == 0:
            ax.set_visible(False)
            continue
        for spk, sst in SPEAKER_STYLE.items():           # each speaker kept visible
            s = d[d.speaker == spk].dropna(subset=[col]).sort_values("alpha")
            if not s.empty:
                ax.plot(s.alpha, s[col] * scale, sst["ls"], color=MUTED, marker=sst["m"],
                        ms=3, lw=0.9, alpha=0.8)
        p = pool_alpha(d, col)                           # pooled mean on top
        if not p.empty:
            ax.fill_between(p.alpha, p.lo * scale, p.hi * scale, color=st["c"],
                            alpha=0.12, linewidth=0)
            ax.plot(p.alpha, p["mean"] * scale, "-", color=st["c"], marker=st["m"],
                    ms=5, lw=2)
        if ref == "native" and col == CEILING_PANEL:
            c = ceiling(long_df, step)
            if c is not None:
                ax.axhline(c * scale, color=INK, lw=1.2, ls=(0, (4, 3)), zorder=1)
                drew_ceiling = True
        ax.set_xlabel("accent strength α")
        finish_panel(ax, title, ylab, arrow)

    handles = [Line2D([], [], color=st["c"], lw=2, marker=st["m"], ms=5,
                      label=f"{st['label']} (speakers pooled)")]
    handles += [Line2D([], [], color=MUTED, lw=0.9, ls=s["ls"], marker=s["m"], ms=3,
                       label=s["label"]) for s in SPEAKER_STYLE.values()]
    if drew_ceiling:
        handles.append(Line2D([], [], color=INK, lw=1.2, ls=(0, (4, 3)),
                              label="L1-prompt ceiling (α = 0)"))
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 3),
               fontsize=9, bbox_to_anchor=(0.5, -0.04))
    fig.suptitle(f"{st['label']} — {REF_LABEL[ref]}, after {step:,} steps",
                 fontsize=12, y=1.0)
    fig.tight_layout(rect=(0, 0.02, 1, 0.98))
    return fig, step


for acc, df in DATA.items():
    fig, step = fig_alpha_single_accent(df, acc, "l1")
    if fig is not None:
        save(fig, f"ch2_{acc}_l1")

### 2.2 Cross-accent overview

One line per accent, each at its own final checkpoint. Four panels only — see
`OVERVIEW_DROP` in §1.4 for why KL-PPG and UTMOS are not on this figure.

In [ ]:
def fig_alpha_all_accents(data, ref, steps=None, panels=None):
    """Metric grid vs α, one line per accent (each at its final checkpoint)."""
    panels = OVERVIEW_PANELS if panels is None else panels
    fig, axes = axes_for(panels)
    used, drew_ceiling = {}, False
    for ax, (col, title, ylab, arrow, scale) in zip(axes, panels):
        drawn = False
        for acc, long_df in data.items():
            d = long_df[long_df.ref_kind == ref]
            if d.empty:
                continue
            step = int(d.step.max()) if not steps else int(steps.get(acc, d.step.max()))
            used[acc] = step
            p = pool_alpha(d[d.step == step], col)
            if p.empty:
                continue
            st = style(acc)
            ax.plot(p.alpha, p["mean"] * scale, color=st["c"], ls=st["ls"],
                    marker=st["m"], ms=4, lw=1.8, label=st["label"])
            if ref == "native" and col == CEILING_PANEL:
                c = ceiling(long_df, step)
                if c is not None:
                    ax.axhline(c * scale, color=st["c"], lw=1.0, ls=(0, (4, 3)),
                               alpha=0.8, zorder=1)
                    drew_ceiling = True
            drawn = True
        if not drawn:
            ax.set_visible(False)
            continue
        ax.set_xlabel("accent strength α")
        finish_panel(ax, title, ylab, arrow)
    for ax in axes[len(panels):]:
        ax.set_visible(False)

    handles = [Line2D([], [], color=style(a)["c"], ls=style(a)["ls"], marker=style(a)["m"],
                      ms=4, lw=1.8, label=f"{style(a)['label']} ({used[a]:,} steps)")
               for a in used]
    if drew_ceiling:
        handles.append(Line2D([], [], color=MUTED, lw=1.0, ls=(0, (4, 3)),
                              label="L1-prompt ceiling (α = 0, per accent)"))
    # Rows of at most three: a single row of five entries is wider than the panel
    # grid, so the tight bbox pads the figure sideways and shrinks the panels.
    legend_rows(fig, handles, per_row=3)
    # No title: the figure is captioned in the text, and the legend already names
    # each accent's final checkpoint.
    fig.tight_layout(rect=(0, 0.02, 1, 1))
    return fig


fig = fig_alpha_all_accents(DATA, "l1")
save(fig, "ch2_all_accents_l1")

### 2.3 Headline numbers

The monotonicity correlations and leakage onsets the eval array records in each
`rq1.csv` footer, at every accent's final checkpoint. `wer_leak_onset` /
`lid_leak_onset` are `NaN` where the signal never crossed threshold anywhere in
the swept α range — that is a censored observation, not a missing one, and §8.1
draws it as such.

In [ ]:
def rq1_summary(data, foot, ref="l1"):
    rows = []
    for acc, f in foot.items():
        step = FINALS[acc]
        s = f[(f.ref_kind == ref) & (f.step == step)]
        if s.empty:
            continue
        row = {"accent": acc, "step": step}
        for c in ("spearman_accent_cs", "spearman_spk_sim", "spearman_wer",
                  "spearman_eng_lid", "wer_leak_onset", "lid_leak_onset"):
            row[c] = s[c].mean() if c in s.columns else np.nan
        d = data[acc]
        d = d[(d.ref_kind == ref) & (d.step == step)]
        peak = d.groupby("alpha")["accent_cs"].mean()
        row["peak_accent_cs_alpha"] = float(peak.idxmax()) if len(peak) else np.nan
        row["accent_cs_at_0"] = float(peak.loc[0.0]) if 0.0 in peak.index else np.nan
        row["accent_cs_peak"] = float(peak.max()) if len(peak) else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("accent")


RQ1_SUMMARY = rq1_summary(DATA, FOOT, "l1")
RQ1_SUMMARY.round(3)

## 3. RQ2a — accent conversion under the GAE prompt

Chapter 3. The same sweeps read against `ref_kind = native`: the prompt is a
General American English clip, so any accent in the output came from the vector
rather than from the reference. The layouts are §2's, with the L1-prompt ceiling
added to the accent-similarity panel.

### 3.1 Per-accent grids and the cross-accent overview

In [ ]:
for acc, df in DATA.items():
    fig, step = fig_alpha_single_accent(df, acc, "native")
    if fig is not None:
        save(fig, f"ch3_{acc}_gae")

fig = fig_alpha_all_accents(DATA, "native")
save(fig, "ch3_all_accents_gae")

### 3.2 The recovery ratio

Per accent, at one checkpoint,

$$R(\alpha) = \frac{\mathrm{CS}_{\mathrm{GAE}}(\alpha) - \mathrm{CS}_{\mathrm{GAE}}(0)}
                   {\mathrm{CS}_{\mathrm{L1}}(0) - \mathrm{CS}_{\mathrm{GAE}}(0)}$$

with CS = AccentCS. $R = 0$ is the GAE-prompt floor (base model, no vector),
$R = 1$ the L1-prompt ceiling of Chapter 2. This is the only place $R$ is
computed: the chapter figure, the operating-point table and the two appendix
tables all come from one pass, so they cannot disagree.

A narrow denominator is the failure mode of a ratio like this — when the L1
prompt buys almost no accent over the GAE prompt, $R$ divides a real difference
by noise and reports a confident-looking multiple. So the gap is carried beside
$R$ everywhere, and $R$ is suppressed wherever the gap falls below `MIN_GAP`.

In [ ]:
# Not a significance test — rq1.csv stores only per-α means over the n utterances,
# so no within-condition SD is available to build one from. It is a stated
# legibility convention, and the gap itself is reported so a reader can re-judge.
MIN_GAP = 0.10


def _cs_at_final(long_df, speaker=None):
    """(step, cs_l1, cs_gae, wer_gae) vs α at the accent's final checkpoint.

    `speaker=None` pools the m/f prompts by averaging AccentCS first and taking the
    ratio of the pooled curves — rather than averaging two per-speaker ratios,
    which would let one near-degenerate denominator dominate the mean.
    """
    d = long_df.dropna(subset=["accent_cs"])
    if speaker is not None:
        d = d[d.speaker == speaker]
    if d.empty:
        return (None,) * 4
    step = int(d.step.max())
    d = d[d.step == step]
    cs = {ref: d[d.ref_kind == ref].groupby("alpha").accent_cs.mean().sort_index()
          for ref in ("l1", "native")}
    if cs["l1"].empty or cs["native"].empty or 0.0 not in cs["native"].index:
        return (None,) * 4
    gae = d[d.ref_kind == "native"]
    wer = (gae.groupby("alpha").wer.mean().sort_index() if "wer" in gae.columns
           else pd.Series(dtype=float))
    return step, cs["l1"], cs["native"], wer


def recovery(long_df, speaker=None, min_gap=MIN_GAP):
    """Per-α recovery for one accent, or None if the sweep is missing.

    `alpha_star` is the best-recovery α and `alpha_int` the best that also keeps
    WER at or below WER_UNUSABLE. Both are the argmax of CS_gae, which is the
    argmax of R whenever R is defined (the two differ by a positive affine map),
    so they stay well-defined for accents whose R is suppressed.
    """
    step, cs_l1, cs_gae, wer = _cs_at_final(long_df, speaker)
    if step is None or 0.0 not in cs_l1.index:
        return None
    ceil_, floor_ = float(cs_l1.loc[0.0]), float(cs_gae.loc[0.0])
    gap = ceil_ - floor_
    valid = gap >= min_gap
    R = (cs_gae - floor_) / gap if valid else pd.Series(np.nan, index=cs_gae.index)
    pos = cs_gae[cs_gae.index > 0]
    ok = pos[[a in wer.index and wer.loc[a] <= WER_UNUSABLE for a in pos.index]]
    return {"step": step, "ceiling": ceil_, "floor": floor_, "gap": gap, "valid": valid,
            "alpha_star": float(pos.idxmax()) if not pos.empty else float("nan"),
            "alpha_int": float(ok.idxmax()) if not ok.empty else float("nan"),
            "cs": cs_gae, "wer": wer, "R": R}


def recovery_long(data, min_gap=MIN_GAP):
    """Tidy per-(accent, prompt, α) frame — the CSV behind every artefact here."""
    rows = []
    for acc, df in data.items():
        for prompt in ("pooled", "m", "f"):
            r = recovery(df, None if prompt == "pooled" else prompt, min_gap)
            if r is None:
                continue
            for a in r["cs"].index:
                rows.append({"accent": acc, "prompt": prompt, "step": r["step"],
                             "alpha": float(a), "cs_gae": float(r["cs"].loc[a]),
                             "recovery": float(r["R"].loc[a]), "ceiling": r["ceiling"],
                             "floor": r["floor"], "gap": r["gap"], "gap_ok": r["valid"],
                             "alpha_star": r["alpha_star"], "alpha_int": r["alpha_int"]})
    return pd.DataFrame(rows)


RECOVERY = {a: recovery(df) for a, df in DATA.items()}
RECOVERY = {a: r for a, r in RECOVERY.items() if r}
RECOVERY_LONG = recovery_long(DATA)
pd.DataFrame([{"accent": a, "step": r["step"], "ceiling": r["ceiling"],
               "floor": r["floor"], "gap": r["gap"], "R reported": r["valid"],
               "α*": r["alpha_star"], "α†": r["alpha_int"]}
              for a, r in RECOVERY.items()]).set_index("accent").round(3)

In [ ]:
def fig_recovery(data, min_gap=MIN_GAP, intervals=False):
    """R vs α, one line per accent.

    `intervals` adds the floor–ceiling panel. It is off by default: the interval is
    the first three columns of the operating-point table and the dashed ceiling of
    §3.1's overview, so drawing it a third time only asks the reader to reconcile
    three views of the same three numbers. Turn it on for a figure that must stand
    alone.
    """
    if intervals:
        fig, (ax, axb) = plt.subplots(1, 2, figsize=(10.5, 4.0),
                                      gridspec_kw={"width_ratios": [1.35, 1]})
    else:
        fig, ax = plt.subplots(figsize=(6.2, 4.0))
        axb = None
    recs = {a: r for a, r in ((a, recovery(df, None, min_gap)) for a, df in data.items()) if r}
    accs = list(recs)

    ax.axhline(1.0, color=MUTED, lw=1.0, ls=(0, (4, 3)))
    ax.axhline(0.0, color=MUTED, lw=1.0)
    for acc in accs:
        r, st = recs[acc], style(acc)
        if not r["valid"]:
            continue
        ax.plot(r["R"].index, r["R"].values, color=st["c"], ls=st["ls"], marker=st["m"],
                ms=4, lw=1.8, label=st["label"])
        a = r["alpha_star"]
        ax.plot([a], [r["R"].loc[a]], marker="o", ms=9, mfc="none", mew=1.4, color=st["c"])
        ai = r["alpha_int"]
        if np.isfinite(ai):
            ax.plot([ai], [r["R"].loc[ai]], marker="o", ms=6, color=st["c"], mew=0)
    ax.set_xlabel("accent strength α")
    ax.set_ylabel("recovery ratio $R(\\alpha)$", fontsize=9)
    ax.set_title("Recovery of the L1-prompt accent  ↑", fontsize=10)
    ax.margins(x=0.04)
    # Labelled on the right: near α=0 every curve is pinned to R=0, so a label
    # there sits on top of the data.
    ax.annotate("L1-prompt ceiling", xy=(1.0, 1.0), xytext=(-2, 3), fontsize=8,
                color=MUTED, textcoords="offset points", va="bottom", ha="right")
    ax.annotate("GAE-prompt floor", xy=(1.0, 0.0), xytext=(-2, 3), fontsize=8,
                color=MUTED, textcoords="offset points", va="bottom", ha="right")

    if axb is not None:
        for y, acc in enumerate(accs):
            r, st = recs[acc], style(acc)
            faint = 1.0 if r["valid"] else 0.35
            axb.plot([r["floor"], r["ceiling"]], [y, y], color=st["c"], lw=3,
                     alpha=0.35 * faint, solid_capstyle="round")
            for x in (r["floor"], r["ceiling"]):
                axb.plot([x], [y], marker="|", ms=11, mew=2, color=st["c"], alpha=faint)
            axb.plot([r["cs"].loc[r["alpha_star"]]], [y], marker="o", ms=8, mfc="none",
                     mew=1.4, color=st["c"], alpha=faint)
            if np.isfinite(r["alpha_int"]):
                axb.plot([r["cs"].loc[r["alpha_int"]]], [y], marker="o", ms=6,
                         color=st["c"], mew=0, alpha=faint)
            if not r["valid"]:
                axb.annotate(f"gap {r['gap']:.2f} < {min_gap:g}: $R$ not reported",
                             xy=(max(r["ceiling"], r["floor"]), y), xytext=(14, 0),
                             textcoords="offset points", fontsize=8, color=MUTED,
                             va="center")
        axb.set_yticks(range(len(accs)))
        axb.set_yticklabels([style(a)["label"] for a in accs], fontsize=9)
        axb.set_ylim(len(accs) - 0.4, -0.6)
        axb.set_xlabel("AccentCS")
        axb.set_title("Floor–ceiling interval, and where α lands", fontsize=10)
        axb.margins(x=0.12)
        axb.grid(axis="y", visible=False)

    handles = [Line2D([], [], color=style(a)["c"], ls=style(a)["ls"], marker=style(a)["m"],
                      ms=4, lw=1.8, label=f"{style(a)['label']} ({recs[a]['step']:,} steps)")
               for a in accs if recs[a]["valid"]]
    # An accent with no line in the left panel must not appear in the legend as
    # though it had one; it keeps its interval in the right panel.
    dropped = [style(a)["label"] for a in accs if not recs[a]["valid"]]
    if dropped:
        handles.append(Line2D([], [], ls="none", marker="",
                              label=f"{', '.join(dropped)}: gap < {min_gap:g}, $R$ undefined"))
    handles += [Line2D([], [], ls="none", marker="o", ms=9, mfc="none", mew=1.4,
                       color=MUTED, label="α* (peak recovery)"),
                Line2D([], [], ls="none", marker="o", ms=6, color=MUTED, mew=0,
                       label=f"α† (peak with WER ≤ {100 * WER_UNUSABLE:.0f}%)")]
    fig.legend(handles=handles, loc="lower center", ncol=3 if intervals else 2,
               fontsize=9, bbox_to_anchor=(0.5, -0.10 if intervals else -0.16))
    fig.tight_layout(rect=(0, 0.02, 1, 1))
    return fig


fig = fig_recovery(DATA)
save(fig, "ch3_recovery_gae")

### 3.3 Operating-point and appendix tables

The main-text table is deliberately narrow. Four things a wider draft carried
are recoverable elsewhere rather than lost: the unconstrained peak α\* and its
R (the ring marker in §3.2, and a row of both appendix tables); UTMOS, which the
overview figures already drop; and the α=0 costs each cost column used to repeat
in brackets — the floors sit in a narrow band across accents, so the caption
states the band once. Everything stays in the CSV.

In [ ]:
TABLE_DIR = RESULTS / "tables"
APPENDIX_DIR = RESULTS / "appendix"
SPK_LABEL = {"m": "male", "f": "female"}


def _at(long_df, step, alpha, col, ref="native", speaker=None):
    """Speaker-pooled value of `col` at one (step, α) of the GAE sweep."""
    d = long_df[(long_df.step == step) & (long_df.ref_kind == ref)]
    if speaker is not None:
        d = d[d.speaker == speaker]
    d = d[np.isclose(d.alpha, alpha)].dropna(subset=[col]) if col in d.columns else d.iloc[:0]
    return float(d[col].mean()) if not d.empty else float("nan")


def _fmt(x, nd=3):
    return "--" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{nd}f}"


def _num(x):
    """Thousands separator that survives LaTeX's maths mode."""
    return f"{int(x):,}".replace(",", "{,}")


def operating_point(data, min_gap=MIN_GAP):
    """One row per accent: the interval, the best α, and what it costs."""
    rows = []
    for acc, df in data.items():
        r = recovery(df, None, min_gap)
        if r is None:
            continue
        a, ai = r["alpha_star"], r["alpha_int"]
        row = {"accent": acc, "step": r["step"], "ceiling": r["ceiling"],
               "floor": r["floor"], "gap": r["gap"], "gap_ok": r["valid"],
               "alpha_star": a, "R_star": float(r["R"].loc[a]), "alpha_int": ai,
               "R_int": float(r["R"].loc[ai]) if np.isfinite(ai) else float("nan")}
        for col in ("wer", "utmos", "spk_sim"):
            row[f"{col}_0"] = _at(df, r["step"], 0.0, col)
            row[f"{col}_star"] = _at(df, r["step"], a, col)
            row[f"{col}_int"] = _at(df, r["step"], ai, col) if np.isfinite(ai) else float("nan")
        rows.append(row)
    return pd.DataFrame(rows)


def operating_point_tex(op, path, min_gap=MIN_GAP):
    body = [
        f"{style(r.accent)['label']} & {_fmt(r.ceiling)} & {_fmt(r.floor)} & {_fmt(r.gap)} & "
        f"{r.alpha_int:.1f} & {_fmt(r.R_int, 2) if r.gap_ok else '--'} & "
        f"{100 * r.wer_int:.1f} & {_fmt(r.spk_sim_int)} \\\\"
        for _, r in op.iterrows()]
    steps = ", ".join(f"{style(r.accent)['label']} {_num(r.step)}" for _, r in op.iterrows())
    tex = f"""% requires: \\usepackage{{booktabs}}
\\begin{{table}}[t]
\\centering\\small
\\caption{{Cross-accent prompting at each accent's final checkpoint ({steps} steps),
  prompt speakers pooled. The ceiling $\\mathrm{{CS}}_{{\\mathrm{{L1}}}}(0)$ and floor
  $\\mathrm{{CS}}_{{\\mathrm{{GAE}}}}(0)$ are the two anchors of Equation~\\ref{{eq:recovery}}
  and the gap between them is what $R$ divides by. Recovery peaks at
  $\\alpha \\geq {op.alpha_star.min():.1f}$ for every accent, where WER is
  {100 * op.wer_star.min():.0f}--{100 * op.wer_star.max():.0f}\\%; the table
  therefore reports $\\alpha^\\dagger$, the greatest recovery that keeps WER at or below
  {100 * WER_UNUSABLE:.0f}\\%, and the two costs paid there. At $\\alpha=0$ every accent
  starts from WER below {100 * op.wer_0.max():.1f}\\% and speaker similarity in
  {op.spk_sim_0.min():.3f}--{op.spk_sim_0.max():.3f}, so the columns are read against
  those. Speaker similarity is measured against the GAE prompt speaker and so is
  expected to fall as the target accent is imposed. $R$ is not reported where the gap
  is below {min_gap:g} AccentCS: the L1 prompt buys too little accent over the GAE
  prompt there for a fraction of the interval to mean anything.}}
\\label{{tab:rq2-operating-point}}
\\begin{{tabular}}{{lrrrrrrr}}
\\toprule
 & \\multicolumn{{3}}{{c}}{{AccentCS anchors}} & \\multicolumn{{2}}{{c}}{{Recovery}} & \\multicolumn{{2}}{{c}}{{Cost at $\\alpha^\\dagger$}} \\\\
\\cmidrule(lr){{2-4}}\\cmidrule(lr){{5-6}}\\cmidrule(lr){{7-8}}
Accent & Ceil. & Floor & Gap & $\\alpha^\\dagger$ & $R(\\alpha^\\dagger)$ & WER (\\%) & Spk.\\ sim. \\\\
\\midrule
""" + "\n".join(body) + """
\\bottomrule
\\end{tabular}
\\end{table}
"""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(tex)
    print(f"wrote {path}")


def appendix_tex(data, speaker, path, min_gap=MIN_GAP):
    """R(α) for one prompt speaker: α down the rows, accents across."""
    recs = {a: recovery(df, speaker, min_gap) for a, df in data.items()}
    accs = [a for a in recs if recs[a]]
    alphas = sorted({float(x) for a in accs for x in recs[a]["cs"].index})

    def row(name, fn, nd=3):
        return f"{name} & " + " & ".join(_fmt(fn(recs[a]), nd) for a in accs) + " \\\\"

    body = [row("Ceiling $\\mathrm{CS}_{\\mathrm{L1}}(0)$", lambda r: r["ceiling"]),
            row("Floor $\\mathrm{CS}_{\\mathrm{GAE}}(0)$", lambda r: r["floor"]),
            row("Gap", lambda r: r["gap"]), "\\midrule"]
    for al in alphas:
        cells = [_fmt(recs[a]["R"].loc[al], 2) if al in recs[a]["R"].index else "--"
                 for a in accs]
        body.append(f"$\\alpha={al:.1f}$ & " + " & ".join(cells) + " \\\\")
    body += ["\\midrule",
             row("$\\alpha^\\star$", lambda r: r["alpha_star"], 1),
             row("$\\alpha^\\dagger$", lambda r: r["alpha_int"], 1)]
    other = "f" if speaker == "m" else "m"
    steps = ", ".join(f"{style(a)['label']} {_num(recs[a]['step'])}" for a in accs)
    tex = f"""% requires: \\usepackage{{booktabs}}
\\begin{{table}}[t]
\\centering\\footnotesize
\\caption{{Recovery ratio $R(\\alpha)$ under the {SPK_LABEL[speaker]} GAE prompt, at each
  accent's final checkpoint ({steps} steps). The anchors of Equation~\\ref{{eq:recovery}}
  are recomputed within this prompt speaker, so the ceiling is the same-speaker
  L1-prompt condition. A dash marks an accent whose ceiling--floor gap is below
  {min_gap:g} AccentCS, where the ratio divides by a difference too small to interpret.
  $\\alpha^\\star$ is the accent strength of peak recovery and $\\alpha^\\dagger$ the peak
  that keeps WER at or below {100 * WER_UNUSABLE:.0f}\\%, both for this prompt speaker.
  Compare with Table~\\ref{{tab:rq2-recovery-{other}}} for the {SPK_LABEL[other]} prompt.}}
\\label{{tab:rq2-recovery-{speaker}}}
\\begin{{tabular}}{{l{'r' * len(accs)}}}
\\toprule
 & {' & '.join(style(a)['label'] for a in accs)} \\\\
\\midrule
""" + "\n".join(body) + """
\\bottomrule
\\end{tabular}
\\end{table}
"""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(tex)
    print(f"wrote {path}")


OP = operating_point(DATA)
if SAVE:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    APPENDIX_DIR.mkdir(parents=True, exist_ok=True)
    RECOVERY_LONG.to_csv(TABLE_DIR / "ch3_recovery.csv", index=False)
    OP.to_csv(TABLE_DIR / "ch3_recovery_operating_point.csv", index=False)
    operating_point_tex(OP, TABLE_DIR / "ch3_recovery_operating_point.tex")
    for spk in ("m", "f"):
        appendix_tex(DATA, spk, APPENDIX_DIR / f"rq2_recovery_prompt_{spk}.tex")
        RECOVERY_LONG[RECOVERY_LONG.prompt == spk].to_csv(
            APPENDIX_DIR / f"rq2_recovery_prompt_{spk}.csv", index=False)

OP.set_index("accent").round(3)

## 4. RQ2b — trajectory: when does each metric stop moving?

Chapter 4. Every checkpoint of every sweep, asking not *what* the metric is but
*how fast it is still changing*. A metric is called settled at the earliest step
from which it stays within ±k·σ of its final value, σ being the evaluation noise
estimated from the metric's own checkpoint-to-checkpoint jitter (§1.5) — so
"settled" means later training moves it no more than the noise does.

### 4.1 Rate of change per metric

In [ ]:
SLOPE_ALPHAS = [0.3, 0.5, 1.0]      # snapped to the run's α grid
SETTLE_K = 2.0                      # width of the settled band, in units of σ
SLOPE_WINDOW = 5                    # checkpoints in the centred local-slope window


def fig_slopes(long_df, accent, ref, alphas=SLOPE_ALPHAS, k_noise=SETTLE_K,
               window=SLOPE_WINDOW):
    """2x3 grid: local slope of each metric vs training step, one line per α."""
    d = long_df[long_df.ref_kind == ref]
    if d.empty or d.step.nunique() < 4:
        print(f"{accent}/{ref}: <4 checkpoints; skipping slope figure")
        return None, []
    avail = sorted(d.alpha.unique())
    picks = sorted(dict.fromkeys(min(avail, key=lambda x: abs(x - a)) for a in alphas))

    rows = []
    fig, axes = panel_axes()
    for ax, (col, title, ylab, arrow, scale) in zip(axes, PANELS):
        if col not in d.columns or d[col].notna().sum() == 0:
            ax.set_visible(False)
            continue
        for (c, ls, mk), a in zip(ALPHA_STYLE, picks):
            g = (d[d.alpha == a].dropna(subset=[col])
                 .groupby("step")[col].mean().reset_index().sort_values("step"))
            if len(g) < 4:
                continue
            vals = g[col].to_numpy() * scale
            steps, sl = rolling_slope(g.step.to_numpy(), vals, window)
            ax.plot(steps, sl, color=c, ls=ls, marker=mk, ms=3.5, lw=1.5, label=f"α={a:g}")
            settled, band, frac = settling_step(g.step.to_numpy(), vals, k_noise)
            if settled is not None:
                j = int(np.where(steps == settled)[0][0])
                ax.plot([settled], [sl[j]], marker="*", ms=12, color=c,
                        markeredgecolor="white", markeredgewidth=0.6, zorder=5, ls="none")
            rows.append({"accent": accent, "ref_kind": ref, "metric": col, "alpha": a,
                         "settles_at": settled, "noise_band": band,
                         "band_frac_of_range": frac, "final_step": int(steps[-1])})
        ax.axhline(0, color=INK, lw=0.9, ls=(0, (1, 2)))
        ax.set_xlabel("training step")
        finish_panel(ax, title, f"Δ {ylab} / 1k steps", arrow)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(3, 3))

    handles = [Line2D([], [], color=c, ls=ls, marker=mk, ms=3.5, lw=1.5, label=f"α={a:g}")
               for (c, ls, mk), a in zip(ALPHA_STYLE, picks)]
    handles.append(Line2D([], [], color=MUTED, marker="*", ms=12, ls="none",
                          label=f"metric settles (stays within ±{k_noise:g}σ of its "
                                f"final value thereafter; σ = eval noise)"))
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 4),
               fontsize=9, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(f"{style(accent)['label']} — rate of change per metric, {REF_LABEL[ref]} "
                 f"({window}-checkpoint local slope)", fontsize=12, y=1.0)
    fig.tight_layout(rect=(0, 0.03, 1, 0.98))
    return fig, rows


SETTLE_ROWS = []
for ref in ("l1", "native"):
    for acc, df in DATA.items():
        fig, rows = fig_slopes(df, acc, ref)
        SETTLE_ROWS += rows
        if fig is not None:
            save(fig, f"ch4_slopes_{acc}_{REF_SLUG[ref]}")

SETTLE = pd.DataFrame(SETTLE_ROWS)
if SAVE and not SETTLE.empty:
    (FIGDIR / "dissertation").mkdir(parents=True, exist_ok=True)
    SETTLE.to_csv(FIGDIR / "dissertation" / "ch4_stabilisation.csv", index=False)
    print(f"wrote {(FIGDIR / 'dissertation' / 'ch4_stabilisation.csv').relative_to(ROOT)}")

### 4.2 Where each metric settles

One row per metric, one marker per accent. An open right-pointing marker at the
last checkpoint means the metric was *still moving* when training stopped — a
censored observation, not a missing one.

In [ ]:
SUMMARY_ALPHA = 0.5


def fig_stabilisation(settle, ref, alpha):
    df = settle[(settle.ref_kind == ref) & (settle.alpha == alpha)]
    if df.empty:
        print(f"no stabilisation rows for {ref} @ α={alpha}; skipping")
        return None
    metrics = [p[0] for p in PANELS if p[0] in set(df.metric)]
    labels = {p[0]: p[1] for p in PANELS}
    fig, ax = plt.subplots(figsize=(7.5, 0.62 * len(metrics) + 2.4))
    accs = sorted(df.accent.unique())
    # small vertical offset per accent so co-located settling steps stay countable
    dy = {a: (i - (len(accs) - 1) / 2) * 0.16 for i, a in enumerate(accs)}
    for acc, sub in df.groupby("accent"):
        st = style(acc)
        xs, ys, cens = [], [], []
        for m, s_, fin in zip(sub.metric, sub.settles_at, sub.final_step):
            if m not in metrics:
                continue
            y = metrics.index(m) + dy[acc]
            if pd.isna(s_):
                cens.append((fin, y))
            else:
                xs.append(s_)
                ys.append(y)
        ax.plot(xs, ys, ls="none", marker=st["m"], ms=8, color=st["c"], label=st["label"])
        if cens:
            cx, cy = zip(*cens)
            ax.plot(cx, cy, ls="none", marker=">", ms=8, mfc="none", color=st["c"])
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels([labels[m] for m in metrics], fontsize=13)
    ax.set_ylim(len(metrics) - 0.5, -0.7)
    ax.set_xlabel("training step at which the metric stops moving", fontsize=13)
    # k-suffixed ticks rather than a shared 1e3 exponent: at this type size the
    # offset text collides with the axis label.
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:g}k" if v else "0"))
    ax.tick_params(axis="x", labelsize=12)
    handles, _ = ax.get_legend_handles_labels()
    handles.append(Line2D([], [], ls="none", marker=">", ms=8, mfc="none", color=MUTED,
                          label="still moving at the final checkpoint"))
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 3),
               fontsize=12, bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout(rect=(0, 0.02, 1, 1))
    return fig


if not SETTLE.empty:
    alpha = min(sorted(SETTLE.alpha.unique()), key=lambda x: abs(x - SUMMARY_ALPHA))
    for ref in ("l1", "native"):
        fig = fig_stabilisation(SETTLE, ref, alpha)
        if fig is not None:
            save(fig, f"ch4_stabilisation_{REF_SLUG[ref]}")

### 4.3 Raw step curves (exploratory)

The levels behind §4.1's derivatives: metric against training step, one line per
α. Not a chapter figure — this is the view to check when a slope panel looks
surprising. Off by default; set `STEP_CURVES` to an accent name to draw it.

In [ ]:
STEP_CURVES = None       # e.g. "dutch"


def fig_step_curves(long_df, accent, ref, cols=None):
    """Metric vs training step, one line per α, speakers pooled."""
    cols = cols or [p[0] for p in PANELS]
    d = long_df[long_df.ref_kind == ref]
    cols = [c for c in cols if c in d.columns and d[c].notna().any()]
    fig, axes = panel_axes(-(-len(cols) // 3), 3, w=3.4, h=2.8)
    for ax, col in zip(axes, cols):
        g = d.dropna(subset=[col]).groupby(["alpha", "step"])[col].mean().reset_index()
        for a, sub in g.groupby("alpha"):
            sub = sub.sort_values("step")
            ax.plot(sub.step, sub[col], marker="o", ms=3, lw=1.2, label=f"α={a:g}")
        ax.set_xlabel("training step")
        ax.set_title(col, fontsize=10)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    axes[0].legend(fontsize=6, ncol=2)
    fig.suptitle(f"{style(accent)['label']} — {REF_LABEL[ref]}: levels behind the slopes",
                 fontsize=12, y=1.0)
    fig.tight_layout()
    return fig


if STEP_CURVES:
    fig = fig_step_curves(DATA[STEP_CURVES], STEP_CURVES, "l1")
    save(fig, f"step_curves_{STEP_CURVES}_l1", subdir="diagnostics")

## 5. RQ3 — segmental vs prosodic transfer

Chapter 4. **This section is on the corrected scale.** The retired version put a
distance ratio (`seg_closure`) and a signed scalar projection
(`supra_closure_mean`) on one axis as though 1.0 meant the same thing in both.
It does not, and the composite is an unbounded mean of ratios that one
near-degenerate denominator can dominate — §8.4 shows Dutch `f0_mean_closure`
reaching 27.7 and dragging the composite with it.

Both channels are rebuilt here as a **distance to the natural target divided by
its own α=0 baseline**, so they mean the same thing:

$$r(\alpha) = \frac{d(\alpha)}{d(0)} \qquad
  1 = \text{no movement}, \quad <1 = \text{toward natural}, \quad >1 = \text{away}$$

| channel | $d$ |
|---|---|
| segmental | mean symmetric PPG-KL to natural (already in `rq3.csv`) |
| prosodic | RMS over the six scaled descriptors of $(x_\alpha - x_{\text{nat}}) / s_f$ |

$s_f$ is the SD of the α=0 baseline for that descriptor pooled over every
(accent, ref_kind, speaker, step) — a between-condition SD standing in for the
within-natural-speech SD, which `rq3.csv` does not store. Documented, not
hidden. The contour metrics (f0_rmse in cents, the PPG-KL warping path) need a
re-run of the eval and are deliberately absent.

### 5.1 Recovering the natural target

`rq3.csv` stores the synthesised value $x_\alpha$ and the closure
$c_\alpha = (x_\alpha - x_0)/(x_{\text{nat}} - x_0)$ but not $x_{\text{nat}}$
itself. It is exactly recoverable — $x_{\text{nat}} = x_0 + (x_\alpha - x_0)/c_\alpha$
— and every non-degenerate α gives the same answer, so take the median for
numerical safety.

In [ ]:
def recover_natural(df):
    """{(ref_kind, speaker, step, feature): x_natural} for the six supra features."""
    out = {}
    for (ref, spk, step), g in df.groupby(["ref_kind", "speaker", "step"]):
        g = g.sort_values("alpha")
        for feat in SUPRA:
            cc = f"{feat}_closure"
            if feat not in g or cc not in g:
                continue
            base = g[feat].iloc[0]
            ests = [base + (x - base) / c
                    for x, c in zip(g[feat].iloc[1:], g[cc].iloc[1:])
                    if pd.notna(c) and abs(c) > 1e-9 and pd.notna(x)]
            if ests:
                out[(ref, spk, step, feat)] = float(np.median(ests))
    return out


def feature_scales(data):
    """SD of each descriptor's α=0 baseline, pooled over every condition."""
    base = {f: [] for f in SUPRA}
    for df in data.values():
        for _, g in df.groupby(["ref_kind", "speaker", "step"]):
            g = g.sort_values("alpha")
            for f in SUPRA:
                if f in g and pd.notna(g[f].iloc[0]):
                    base[f].append(float(g[f].iloc[0]))
    return {f: (float(np.std(v)) if len(v) > 1 and np.std(v) > 0 else np.nan)
            for f, v in base.items()}


def relative_distances(df, scales):
    """Add seg_rel and sup_rel: distance-to-natural at α over the same at α=0."""
    nat = recover_natural(df)
    df = df.copy()
    df["seg_rel"] = np.nan
    df["sup_rel"] = np.nan
    for (ref, spk, step), g in df.groupby(["ref_kind", "speaker", "step"]):
        g = g.sort_values("alpha")
        idx = g.index
        kl = g["seg_ppg_kl_to_natural"].to_numpy(dtype=float)
        if np.isfinite(kl).all() and kl[0] > 0:
            df.loc[idx, "seg_rel"] = kl / kl[0]
        d = []
        for _, row in g.iterrows():
            terms = []
            for f in SUPRA:
                xn = nat.get((ref, spk, step, f))
                s = scales.get(f)
                if xn is None or s is None or not np.isfinite(s) or pd.isna(row.get(f)):
                    continue
                terms.append(((row[f] - xn) / s) ** 2)
            d.append(np.sqrt(np.mean(terms)) if terms else np.nan)
        d = np.asarray(d, dtype=float)
        if np.isfinite(d).all() and d[0] > 0:
            df.loc[idx, "sup_rel"] = d / d[0]
    return df, nat


# Output path kept as-is: the chapter's \includegraphics already point here.
RQ3_DIR = "rq3_v2"

SCALES = feature_scales(DATA)
print("per-feature scales (SD of the α=0 baseline, pooled over conditions):")
for f, s in SCALES.items():
    print(f"  {f:<14} {s:.4g}  ({SUPRA_UNIT[f]})")

DATA3, NATURAL = {}, {}
for acc, df in DATA.items():
    DATA3[acc], NATURAL[acc] = relative_distances(df, SCALES)

### 5.2 Relative distance per accent

Error bars are ±2σ from the checkpoint-to-checkpoint jitter at that α. The
shaded region marks where WER has passed 20% and the output is no longer usable
English, so movement there is bought at a price the chapter does not accept.

In [ ]:
def fig_relative(data3, finals, ref):
    """Accent small multiples. y = d(α)/d(0), log. 1.0 = the vector moved nothing."""
    accs = [a for a in data3 if not pooled(data3[a], "seg_rel", ref, finals[a]).empty]
    if not accs:
        print(f"no seg_rel for ref={ref}; skipping")
        return None
    ncol = 2
    nrow = int(np.ceil(len(accs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.15 * nrow),
                             sharex=True, sharey=True, squeeze=False)
    flat = [ax for r in axes for ax in r]
    for ax, acc in zip(flat, accs):
        df, step = data3[acc], finals[acc]
        la = leak_alpha(df, ref, step)
        if np.isfinite(la):
            ax.axvspan(la, 1.02, color=INK, alpha=.055, lw=0)
            ax.text(min(la + .02, .97), 3.4, "WER > 20%", fontsize=11, color=MUTED,
                    ha="left", va="top")
        ax.axhline(1.0, color=INK, lw=1.0, ls=(0, (1, 2)))
        for col, c, ls, mk, lab in (("sup_rel", SUP_C, "--", "s", "prosodic"),
                                    ("seg_rel", SEG_C, "-", "o", "segmental")):
            p = pooled(df, col, ref, step)
            if p.empty:
                continue
            errs = [2 * (noise_sd(df[(df.ref_kind == ref) & (df.alpha == a)]
                                  .groupby("step")[col].mean().sort_index().to_numpy())
                         or np.nan) for a in p.alpha]
            ax.errorbar(p.alpha, p["mean"], yerr=errs, fmt=ls, color=c, marker=mk,
                        ms=4.5, lw=1.9, elinewidth=1.0, capsize=2.4, label=lab)
        ratio_axis(ax, "y", (0.5, 1, 2, 4), (0.42, 4.6))
        ax.set_title(style(acc)["label"], fontsize=14)
        ax.margins(x=.04)
    n = len(accs)
    # spare slots: blanked but not hidden, so one can host the legend
    for ax in flat[n:]:
        ax.axis("off")
    # The last DATA axis in each column carries the x-axis; with an odd accent
    # count the bottom row is short, so axes[-1] alone would leave a column mute.
    for c in range(ncol):
        idxs = [r * ncol + c for r in range(nrow) if r * ncol + c < n]
        if idxs:
            flat[idxs[-1]].set_xlabel("accent strength α")
            flat[idxs[-1]].tick_params(labelbottom=True)
    for r in axes:
        r[0].set_ylabel("relative distance")
    handles = [Line2D([], [], color=SEG_C, ls="-", marker="o", ms=4.5, lw=1.9,
                      label="Segmental — PPG-KL to natural"),
               Line2D([], [], color=SUP_C, ls="--", marker="s", ms=4.5, lw=1.9,
                      label="Prosodic — scaled descriptor distance"),
               Line2D([], [], color=INK, ls=(0, (1, 2)), lw=1.0,
                      label="1.0 = the vector moved nothing")]
    if n < len(flat):
        flat[n].legend(handles=handles, loc="center", fontsize=12, frameon=False,
                       handlelength=2.8, labelspacing=1.0)
        fig.tight_layout()
    else:
        fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=12,
                   bbox_to_anchor=(0.5, -0.07))
        fig.tight_layout(rect=(0, 0.02, 1, 1))
    return fig


for ref in ("l1", "native"):
    fig = fig_relative(DATA3, FINALS, ref)
    if fig is not None:
        save(fig, f"fig_rq3_relative_{REF_SLUG[ref]}", subdir=RQ3_DIR)

### 5.3 Both prompts on one panel set

The same quantity with colour = channel and line style / marker fill = prompt.
Each prompt's leakage onset is a vertical rule in its own style, replacing the
shaded band that two overlapping conditions would make unreadable. Use this
where the chapter needs one figure instead of the pair in §5.2.

In [ ]:
def fig_relative_both(data3, finals):
    accs = [a for a in data3
            if not (pooled(data3[a], "seg_rel", "l1", finals[a]).empty
                    and pooled(data3[a], "seg_rel", "native", finals[a]).empty)]
    if not accs:
        print("nothing to plot")
        return None
    ncol = 2
    nrow = int(np.ceil((len(accs) + 1) / ncol))      # +1 reserves a legend slot
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.15 * nrow),
                             sharex=True, sharey=True, squeeze=False)
    flat = [ax for r in axes for ax in r]
    REFS = (("l1", "-", True), ("native", "--", False))
    for ax, acc in zip(flat, accs):
        df, step = data3[acc], finals[acc]
        ax.axhline(1.0, color=INK, lw=1.0, ls=(0, (1, 2)), zorder=1)
        for ref, ls, filled in REFS:
            la = leak_alpha(df, ref, step)
            if np.isfinite(la):
                ax.axvline(la, color=MUTED, ls=ls, lw=1.1, alpha=.7, zorder=1)
            for col, c, mk in (("sup_rel", SUP_C, "s"), ("seg_rel", SEG_C, "o")):
                p = pooled(df, col, ref, step)
                if p.empty:
                    continue
                errs = [2 * (noise_sd(df[(df.ref_kind == ref) & (df.alpha == a)]
                                      .groupby("step")[col].mean().sort_index().to_numpy())
                             or np.nan) for a in p.alpha]
                ax.errorbar(p.alpha, p["mean"], yerr=errs, color=c, ls=ls, marker=mk,
                            ms=4.2, lw=1.7, elinewidth=0.8, capsize=2.0, zorder=3,
                            markerfacecolor=(c if filled else "white"),
                            markeredgecolor=c, markeredgewidth=1.1)
        ratio_axis(ax, "y", (0.5, 1, 2, 4), (0.42, 4.6))
        ax.set_title(style(acc)["label"], fontsize=14)
        ax.margins(x=.04)
    n = len(accs)
    for ax in flat[n:]:
        ax.axis("off")
    for c in range(ncol):
        idxs = [r * ncol + c for r in range(nrow) if r * ncol + c < n]
        if idxs:
            flat[idxs[-1]].set_xlabel("accent strength α")
            flat[idxs[-1]].tick_params(labelbottom=True)
    for r in axes:
        r[0].set_ylabel("relative distance")

    def _h(c, ls, mk, filled, lab):
        return Line2D([], [], color=c, ls=ls, marker=mk, ms=4.2, lw=1.7, label=lab,
                      markerfacecolor=(c if filled else "white"),
                      markeredgecolor=c, markeredgewidth=1.1)

    handles = [_h(SEG_C, "-", "o", True, "Segmental — L1 prompt"),
               _h(SEG_C, "--", "o", False, "Segmental — GAE prompt"),
               _h(SUP_C, "-", "s", True, "Prosodic — L1 prompt"),
               _h(SUP_C, "--", "s", False, "Prosodic — GAE prompt"),
               Line2D([], [], color=MUTED, ls="-", lw=1.1, label="leakage onset (WER > 20%)"),
               Line2D([], [], color=INK, ls=(0, (1, 2)), lw=1.0, label="1.0 = no movement")]
    if n < len(flat):
        flat[n].legend(handles=handles, loc="center", fontsize=11, frameon=False,
                       handlelength=2.8, labelspacing=.9)
        fig.tight_layout()
    else:
        fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=11,
                   bbox_to_anchor=(0.5, -0.07))
        fig.tight_layout(rect=(0, 0.02, 1, 1))
    return fig


fig = fig_relative_both(DATA3, FINALS)
if fig is not None:
    save(fig, "fig_rq3_relative_both", subdir=RQ3_DIR)

### 5.4 The movement plane

x = segmental relative distance, y = prosodic. The α=0 baseline sits at (1, 1);
natural speech is the origin. The *direction of travel* is the answer to RQ3.

In [ ]:
def fig_plane(data3, finals):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.5), sharex=True, sharey=True)
    for ax, ref in zip(axes, ("l1", "native")):
        ax.axhline(1, color=GRID, lw=1.0)
        ax.axvline(1, color=GRID, lw=1.0)
        ax.plot([1], [1], marker="+", ms=11, color=INK, mew=1.4, ls="none", zorder=4)
        ax.annotate("α = 0\n(no movement)", (1, 1), textcoords="offset points",
                    xytext=(-8, -22), fontsize=7.5, color=MUTED, ha="right")
        for acc, df in data3.items():
            st, step = style(acc), finals[acc]
            xs = pooled(df, "seg_rel", ref, step)
            ys = pooled(df, "sup_rel", ref, step)
            if xs.empty or ys.empty:
                continue
            m = xs.merge(ys, on="alpha", suffixes=("_s", "_p")).sort_values("alpha")
            ax.plot(m["mean_s"], m["mean_p"], color=st["c"], linestyle=st["ls"],
                    marker=st["m"], ms=4.5, lw=1.6, label=st["label"], zorder=3)
            last = m.iloc[-1]
            ax.annotate("1.0", (last["mean_s"], last["mean_p"]), fontsize=7,
                        color=st["c"], textcoords="offset points", xytext=(4, -3))
        ratio_axis(ax, "x", (0.5, 1, 2, 4), (0.45, 5.0))
        ratio_axis(ax, "y", (0.5, 1, 2, 4), (0.45, 5.0))
        ax.set_xlabel("segmental  ←  toward natural")
        ax.set_title(REF_LABEL[ref], fontsize=10.5)
    axes[0].set_ylabel("prosodic  ↓  toward natural")
    axes[0].legend(loc="upper left", fontsize=8)
    fig.suptitle("RQ3 — the movement plane: which way does the vector travel?  "
                 "(origin = natural speech; markers are α)", fontsize=11.5, y=1.0)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


fig = fig_plane(DATA3, FINALS)
save(fig, "fig_rq3_plane", subdir=RQ3_DIR)

### 5.5 Cross-accent summary at a matched α

In [ ]:
RQ3_ALPHA = 0.5


def fig_rq3_summary(data3, finals, alpha=RQ3_ALPHA):
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.4), sharex=True)
    accs = list(data3)
    for ax, ref in zip(axes, ("l1", "native")):
        ax.axvline(1.0, color=INK, lw=1.0, ls=(0, (1, 2)))
        for i, acc in enumerate(accs):
            df, step = data3[acc], finals[acc]
            a = min(sorted(df.alpha.unique()), key=lambda v: abs(v - alpha))
            sub = df[(df.ref_kind == ref) & (df.step == step) & (df.alpha == a)]
            if sub.empty:
                continue
            s_, p_ = sub["seg_rel"].mean(), sub["sup_rel"].mean()
            if np.isfinite(s_) and np.isfinite(p_):
                ax.plot([min(s_, p_), max(s_, p_)], [i, i], color=GRID, lw=2.4, zorder=1)
            ax.plot([s_], [i], marker="o", ms=8, color=SEG_C, zorder=3)
            ax.plot([p_], [i], marker="s", ms=8, color=SUP_C, zorder=3)
        ax.set_yticks(range(len(accs)))
        ax.set_yticklabels([style(a_)["label"] for a_ in accs], fontsize=9)
        ax.set_ylim(len(accs) - .5, -.6)
        ratio_axis(ax, "x", (0.5, 1, 2), (0.45, 2.3))
        ax.set_xlabel("distance to natural ÷ baseline")
        ax.set_title(REF_LABEL[ref], fontsize=10.5)
    handles = [Line2D([], [], color=SEG_C, marker="o", ms=8, ls="none", label="Segmental"),
               Line2D([], [], color=SUP_C, marker="s", ms=8, ls="none", label="Prosodic"),
               Line2D([], [], color=INK, ls=(0, (1, 2)), lw=1.0, label="no movement")]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=8.5,
               bbox_to_anchor=(0.5, -0.10))
    fig.suptitle(f"RQ3 — both channels at α ≈ {alpha:g}, final checkpoint per accent",
                 fontsize=11.5, y=1.02)
    fig.tight_layout(rect=(0, 0.02, 1, 0.96))
    return fig


fig = fig_rq3_summary(DATA3, FINALS)
save(fig, "fig_rq3_summary", subdir=RQ3_DIR)

### 5.6 Every descriptor in its own units

The relative distances of §5.2 are ratios; this is what they are ratios *of*.
Each descriptor in Hz, %, runs/s or nats, with the α=0 baseline, the natural
target, and the band between them shaded — the distance the vector must travel.

In [ ]:
def fig_gap(df3, nat, acc, step):
    cols = ([("seg_ppg_kl_to_natural", "Segmental PPG-KL to natural", "sym. KL (nats)")]
            + [(f, SUPRA_LABEL[f], SUPRA_UNIT[f]) for f in SUPRA])
    fig, axes = plt.subplots(2, 4, figsize=(13.2, 5.6), squeeze=False)
    flat = [ax for r in axes for ax in r]
    for ax, (col, lab, unit) in zip(flat, cols):
        for ref, c, ls, mk in (("l1", "#0072B2", "-", "o"), ("native", "#D55E00", "--", "s")):
            p = pooled(df3, col, ref, step)
            if p.empty:
                continue
            ax.plot(p.alpha, p["mean"], ls, color=c, marker=mk, ms=3.8, lw=1.7)
            if col in SUPRA:
                base = float(p["mean"].iloc[0])
                tgt = np.nanmean([v for (r_, s_, st_, f_), v in nat.items()
                                  if r_ == ref and st_ == step and f_ == col] or [np.nan])
                if np.isfinite(tgt):
                    lo, hi = sorted((base, tgt))
                    ax.axhspan(lo, hi, color=c, alpha=.10, lw=0)
                    ax.axhline(tgt, color=c, lw=1.0, ls=":")
        ax.set_title(lab, fontsize=8.5)
        ax.set_xlabel("α")
        ax.set_ylabel(unit, fontsize=7.5)
        ax.margins(x=.04)
    for ax in flat[len(cols):]:
        ax.set_visible(False)
    handles = [Line2D([], [], color="#0072B2", ls="-", marker="o", ms=3.8, label="L1 prompt"),
               Line2D([], [], color="#D55E00", ls="--", marker="s", ms=3.8, label="GAE prompt"),
               Line2D([], [], color=MUTED, ls=":", lw=1.0,
                      label="natural target (shaded = the gap)")]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=8.5,
               bbox_to_anchor=(0.5, -0.03))
    fig.suptitle(f"RQ3 — {style(acc)['label']}: every descriptor in its own units, "
                 f"with the gap to close  ({step:,} steps)", fontsize=11.5, y=1.01)
    fig.tight_layout(rect=(0, 0.03, 1, 0.98))
    return fig


for acc in DATA3:
    fig = fig_gap(DATA3[acc], NATURAL[acc], acc, FINALS[acc])
    save(fig, f"fig_rq3_gap_{acc}", subdir=RQ3_DIR)

### 5.7 Trend or checkpoint noise?

step × α heatmap of each channel's relative distance. A real effect is a smooth
field; noise is speckle. The colour scale is symmetric in *log* space, since
ratios are multiplicative — otherwise a channel moving toward natural saturates
at one end.

In [ ]:
def fig_heat(df3, acc):
    steps = sorted(df3.step.unique())
    alphas = sorted(df3.alpha.unique())
    if len(steps) < 3:
        print(f"{acc}: <3 checkpoints; skipping heatmap")
        return None
    rows = [("seg_rel", "Segmental"), ("sup_rel", "Prosodic")]
    fig, axes = plt.subplots(2, 2, figsize=(8.6, 5.4), squeeze=False, layout="constrained")
    for r, (col, lab) in enumerate(rows):
        vals = df3[col].dropna()
        if vals.empty:
            continue
        hi = float(np.nanpercentile(vals, 98))
        lo = float(np.nanpercentile(vals, 2))
        vmax = max(hi, 1.0 / lo if lo > 0 else hi)
        for c, ref in enumerate(("l1", "native")):
            sub = df3[df3.ref_kind == ref]
            M = np.full((len(steps), len(alphas)), np.nan)
            piv = sub.groupby(["step", "alpha"])[col].mean()
            for i, s in enumerate(steps):
                for j, a in enumerate(alphas):
                    if (s, a) in piv.index:
                        M[i, j] = piv.loc[(s, a)]
            ax = axes[r][c]
            im = ax.imshow(np.log10(M), aspect="auto", origin="lower", cmap="RdBu_r",
                           vmin=-np.log10(vmax), vmax=np.log10(vmax))
            ax.set_xticks(range(len(alphas)),
                          [f"{a:g}" for a in alphas] if r == 1 else [], fontsize=7)
            ax.set_yticks(range(len(steps)),
                          [f"{s // 1000}k" for s in steps] if c == 0 else [], fontsize=7)
            ax.grid(False)
            ax.set_title(f"{lab} — {REF_LABEL[ref]}", fontsize=8.5)
            if r == 1:
                ax.set_xlabel("α")
            if c == 0:
                ax.set_ylabel("training step")
        cb = fig.colorbar(im, ax=list(axes[r]), fraction=.035, pad=.015)
        cb.ax.tick_params(labelsize=7)
        cb.outline.set_visible(False)
        ticks = [1 / vmax, 1 / np.sqrt(vmax), 1.0, np.sqrt(vmax), vmax]
        cb.set_ticks(np.log10(ticks))
        cb.set_ticklabels([f"{t:.2f}" for t in ticks])
        cb.set_label("d(α)/d(0)", fontsize=7.5)
    fig.suptitle(f"RQ3 — {style(acc)['label']}: relative distance across every "
                 f"checkpoint × α  (white = 1.0 = no movement)", fontsize=11)
    return fig


for acc in DATA3:
    fig = fig_heat(DATA3[acc], acc)
    if fig is not None:
        save(fig, f"fig_rq3_heat_{acc}", subdir=RQ3_DIR)

### 5.8 The corrected-scale table

In [ ]:
rows = []
for acc, df in DATA3.items():
    for ref in ("l1", "native"):
        step = FINALS[acc]
        sub = df[(df.ref_kind == ref) & (df.step == step)]
        for al in sorted(sub.alpha.unique()):
            s = sub[sub.alpha == al]
            rows.append(dict(accent=acc, ref_kind=ref, step=step, alpha=al,
                             seg_rel=s["seg_rel"].mean(), sup_rel=s["sup_rel"].mean(),
                             wer=s["wer"].mean() if "wer" in s else np.nan))
RQ3_TIDY = pd.DataFrame(rows)
if SAVE:
    (FIGDIR / RQ3_DIR).mkdir(parents=True, exist_ok=True)
    RQ3_TIDY.to_csv(FIGDIR / RQ3_DIR / "rq3_relative.csv", index=False)
    print(f"wrote {(FIGDIR / RQ3_DIR / 'rq3_relative.csv').relative_to(ROOT)}")

RQ3_TIDY[RQ3_TIDY.alpha.isin([0.0, 0.5, 1.0])].pivot_table(
    index=["accent", "ref_kind"], columns="alpha",
    values=["seg_rel", "sup_rel"]).round(3)

## 6. Listening test — subjective validation

Chapter 2. Reads the **fielded** ratings, not a simulation:
`listening-tests/analysis/out/listening_long.csv`, written by
`analyse_listening.py` from the Qualtrics exports. That script keeps the job the
notebook cannot do (parsing two differently-prefixed rating blocks out of the
raw exports); everything downstream of the tidy CSV lives here.

The design as fielded: block A / L1 prompt only, α ∈ {0, 0.25, 0.5, 0.75, 1},
ten utterances across a male and a female speaker, rating = **similarity to the
reference accent** on a 0–100 slider. The natural-speech anchor was played as a
reference but *not rated*, so there is no ceiling value — any figure drawing one
is drawing a number that was never collected.

In [ ]:
from scipy import stats

if not LISTENING_CSV.exists():
    print(f"no ratings at {LISTENING_CSV} — skipping §6")
    RATINGS = None
else:
    RATINGS = pd.read_csv(LISTENING_CSV)
    print(f"{len(RATINGS):,} ratings — {RATINGS.participant.nunique()} participants, "
          f"accents {sorted(RATINGS.accent.unique())}, "
          f"blocks {sorted(RATINGS.block.unique())}, "
          f"conditions {sorted(RATINGS.condition.unique())}, "
          f"α {sorted(RATINGS.alpha.unique())}")

### 6.1 Post-screening, CIs and within-participant tests

Ratings within a participant are dependent, so the participant is the unit
throughout: aggregate to one value per participant per α *first*, then screen,
then test. Screening is MUSHRA-style — drop raters whose 5-condition profile
does not track the group's, with the correlation computed leave-one-out so a
rater cannot inflate the mean they are compared against.

In [ ]:
SCREEN_THRESHOLD = 0.0      # None disables post-screening


def ci95(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    return np.nan if n < 2 else stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)


def screen(per_pa, threshold):
    """(kept, dropped) participant ids by leave-one-out profile correlation."""
    wide = per_pa.pivot(index="participant", columns="alpha", values="rating")
    kept, dropped = [], []
    for pid, row in wide.iterrows():
        others = wide.drop(index=pid).mean(axis=0)
        ok = row.notna() & others.notna()
        if ok.sum() < 3 or row[ok].std() == 0:
            dropped.append(pid)
            continue
        r = stats.spearmanr(row[ok], others[ok]).statistic
        (kept if np.isfinite(r) and r > threshold else dropped).append(pid)
    return kept, dropped


def analyse_accent(accent, long_df, threshold=SCREEN_THRESHOLD):
    """Per-accent summary + the wide (participant × α) frame the tests run on."""
    per_pa = long_df.groupby(["participant", "alpha"], as_index=False)["rating"].mean()
    n_before = per_pa.participant.nunique()
    dropped = []
    if threshold is not None:
        kept, dropped = screen(per_pa, threshold)
        per_pa = per_pa[per_pa.participant.isin(kept)]
    wide = per_pa.pivot(index="participant", columns="alpha", values="rating").dropna()
    alphas = list(wide.columns)

    s = pd.DataFrame([dict(accent=accent, alpha=a, n=len(wide[a]), mean=wide[a].mean(),
                           sd=wide[a].std(ddof=1),
                           se=wide[a].std(ddof=1) / np.sqrt(len(wide[a])),
                           ci95=ci95(wide[a].to_numpy())) for a in alphas])
    s["ci_lo"], s["ci_hi"] = s["mean"] - s["ci95"], s["mean"] + s["ci95"]

    print(f"\n=== {accent} ===")
    print(f"  participants: {n_before} finished -> {wide.shape[0]} retained "
          f"({len(dropped)} excluded by post-screening)")
    peak = s.loc[s["mean"].idxmax()]
    print(f"  peak rating at α = {peak.alpha:g}  ({peak['mean']:.1f})")

    # Omnibus: Friedman (non-parametric repeated measures over the five conditions)
    chi2, p = stats.friedmanchisquare(*[wide[a].to_numpy() for a in alphas])
    print(f"  Friedman: chi2({len(alphas) - 1}) = {chi2:.2f}, p = {p:.2g}")

    # Post-hoc: each α vs α=0, Wilcoxon signed-rank, Holm-corrected
    ref = alphas[0]
    raw = [(a, stats.wilcoxon(wide[a], wide[ref]).pvalue) for a in alphas[1:]]
    order = np.argsort([q for _, q in raw])
    holm, m, running = {}, len(raw), 0.0
    for rank, idx in enumerate(order):
        a, q = raw[idx]
        running = max(running, min(1.0, q * (m - rank)))
        holm[a] = running
    tests = []
    for a, q in raw:
        d = wide[a].mean() - wide[ref].mean()
        print(f"    α {a:g} vs {ref:g}: delta = {d:+.1f}, p_raw = {q:.3g}, "
              f"p_holm = {holm[a]:.3g}{'  *' if holm[a] < 0.05 else ''}")
        tests.append(dict(accent=accent, alpha=a, delta=d, p_raw=q, p_holm=holm[a]))

    # Rater agreement: mean pairwise Spearman between 5-condition profiles.
    prof = wide.to_numpy()
    rs = [stats.spearmanr(prof[i], prof[j]).statistic
          for i in range(len(prof)) for j in range(i + 1, len(prof))]
    rs = [r for r in rs if np.isfinite(r)]
    if rs:
        print(f"  mean pairwise inter-rater Spearman: {np.mean(rs):.2f}")
    return s, wide, pd.DataFrame(tests)


LISTEN_SUMMARY, LISTEN_WIDE, LISTEN_TESTS = None, {}, None
if RATINGS is not None:
    order = [a for a in ACCENTS if a in set(RATINGS.accent)]
    summaries, tests = [], []
    for acc in order:
        s, wide, t = analyse_accent(acc, RATINGS[RATINGS.accent == acc])
        summaries.append(s)
        LISTEN_WIDE[acc] = wide
        tests.append(t)
    LISTEN_SUMMARY = pd.concat(summaries, ignore_index=True)
    LISTEN_TESTS = pd.concat(tests, ignore_index=True)

### 6.2 Mean rating and the paired contrast

The left panel is the raw level; the right is the **paired** difference from
α = 0, which is the quantity the Wilcoxon tests actually evaluate. Independent
CIs on the left overlap even where a within-participant contrast is significant,
so the right panel is what a reader should judge differences from.

In [ ]:
def fig_listening(summary, wides):
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.6))
    for acc in wides:
        st = style(acc)
        d = summary[summary.accent == acc].sort_values("alpha")
        n = int(d["n"].iloc[0])
        axes[0].errorbar(d.alpha, d["mean"], yerr=d["ci95"], color=st["c"], ls=st["ls"],
                         marker=st["m"], ms=5, lw=1.8, capsize=3, elinewidth=1.0,
                         label=f"{st['label']} (n={n})")
        w = wides[acc]
        diff = w.sub(w[w.columns[0]], axis=0)      # paired differences from α = 0
        axes[1].errorbar(diff.columns, diff.mean(axis=0), yerr=diff.apply(ci95, axis=0),
                         color=st["c"], ls=st["ls"], marker=st["m"], ms=5, lw=1.8,
                         capsize=3, elinewidth=1.0)
    lo = min(summary.ci_lo.min(), 0) - 4
    axes[0].set_ylim(max(0, lo), min(100, summary.ci_hi.max() + 4))
    axes[0].set_ylabel("similarity to reference accent")
    axes[0].set_title("Mean rating", fontsize=12)
    axes[0].legend(fontsize=9)
    axes[1].axhline(0, color=INK, lw=1.0, ls=(0, (1, 2)))
    axes[1].set_ylabel("change from α = 0 (paired)")
    axes[1].set_title("Paired difference from α = 0", fontsize=12)
    for ax in axes:
        ax.set_xlabel("accent strength α")
    fig.tight_layout()
    return fig


if LISTEN_SUMMARY is not None:
    fig = fig_listening(LISTEN_SUMMARY, LISTEN_WIDE)
    save(fig, "ch2_listening", subdir="listening")
    if SAVE:
        (FIGDIR / "listening").mkdir(parents=True, exist_ok=True)
        LISTEN_SUMMARY.to_csv(FIGDIR / "listening" / "ch2_listening_summary.csv", index=False)
        LISTEN_TESTS.to_csv(FIGDIR / "listening" / "ch2_listening_tests.csv", index=False)
        print("wrote ch2_listening_summary.csv + ch2_listening_tests.csv")

### 6.3 Subjective against objective

Does the 0–100 slider agree with GenAID? One point per (accent, α) cell, with
AccentCS read from the *result CSVs* at the same L1-prompt final checkpoint —
not from a transcribed table, so it cannot fall out of date.

The pooled ρ is inflated by between-accent offsets (accents sit at different
AccentCS levels for reasons unrelated to α); the within-accent ρ is the quantity
that says whether the two scales agree about the sweep.

In [ ]:
def objective_accent_cs(data, finals, accents, alphas, ref="l1"):
    """{accent: [AccentCS at each listening-test α]}, snapped to the run's α grid."""
    out = {}
    for acc in accents:
        if acc not in data:
            continue
        d = data[acc]
        d = d[(d.ref_kind == ref) & (d.step == finals[acc])]
        cs = d.groupby("alpha")["accent_cs"].mean().sort_index()
        if cs.empty:
            continue
        out[acc] = [float(cs.loc[min(cs.index, key=lambda v: abs(v - a))]) for a in alphas]
    return out


def fig_validation(summary, obj, alphas):
    fig, ax = plt.subplots(figsize=(5.8, 4.8))
    xs, ys, within = [], [], []
    for acc, cs in obj.items():
        st = style(acc)
        s = summary[summary.accent == acc].sort_values("alpha")
        if len(s) != len(cs):
            continue
        sizes = 26 + 130 * np.asarray(alphas)          # α as the redundant cue
        ax.plot(cs, s["mean"], color=st["c"], ls=st["ls"], lw=1.0, alpha=0.55, zorder=2)
        r_in = stats.spearmanr(cs, s["mean"]).statistic
        within.append(r_in)
        ax.scatter(cs, s["mean"], s=sizes, marker=st["m"], facecolor=st["c"],
                   edgecolor="white", lw=1.0, zorder=3,
                   label=rf"{st['label']}  ($\rho$ = {r_in:+.2f})")
        xs += list(cs)
        ys += list(s["mean"])
    rho, p = stats.spearmanr(xs, ys)
    ax.annotate(rf"pooled $\rho$ = {rho:+.2f} (p = {p:.3f})" "\n"
                rf"mean within-accent $\rho$ = {np.mean(within):+.2f}",
                (0.03, 0.96), xycoords="axes fraction", va="top", fontsize=9, color=INK)
    ax.set_xlabel("AccentCS (GenAID embedding cosine)", fontsize=9)
    ax.set_ylabel("mean rating (0–100)", fontsize=9)
    ax.set_title(r"Subjective vs objective accent, per ($\alpha$, accent) cell", fontsize=10)
    ax.legend(fontsize=8, loc="lower right",
              title=r"marker size $\propto\ \alpha$;  $\rho$ = within-accent",
              title_fontsize=8)
    fig.tight_layout()
    return fig


if LISTEN_SUMMARY is not None:
    la = sorted(LISTEN_SUMMARY.alpha.unique())
    OBJ_CS = objective_accent_cs(DATA, FINALS, list(LISTEN_WIDE), la)
    fig = fig_validation(LISTEN_SUMMARY, OBJ_CS, la)
    save(fig, "ch2_listening_validation", subdir="listening")

### 6.4 Rating spread per α

Where the mean is doing the work of a distribution that is not unimodal, the
mean is the wrong summary. One violin per α, with the raw ratings behind it.

In [ ]:
def fig_spread(ratings, accents):
    alphas = sorted(ratings.alpha.unique())
    fig, axes = plt.subplots(1, len(accents), figsize=(2.7 * len(accents), 3.6), sharey=True)
    axes = np.atleast_1d(axes)
    rng = np.random.default_rng(1)
    for ax, acc in zip(axes, accents):
        st = style(acc)
        data = [ratings[(ratings.accent == acc) & (ratings.alpha == a)]["rating"].to_numpy()
                for a in alphas]
        parts = ax.violinplot(data, positions=range(len(alphas)), widths=0.8,
                              showextrema=False, showmedians=False)
        for b in parts["bodies"]:
            b.set_facecolor(st["c"])
            b.set_alpha(0.28)
            b.set_edgecolor("none")
        for i, d in enumerate(data):
            ax.scatter(i + rng.normal(0, 0.055, len(d)), d, s=3, color=st["c"],
                       alpha=0.35, lw=0)
            ax.plot([i - 0.3, i + 0.3], [d.mean()] * 2, color=INK, lw=1.6, zorder=5)
        ax.set_xticks(range(len(alphas)))
        ax.set_xticklabels([f"{a:g}" for a in alphas])
        ax.set_title(st["label"], fontsize=10)
        ax.set_xlabel(r"$\alpha$", fontsize=9)
        ax.set_ylim(0, 100)
    axes[0].set_ylabel("similarity to reference accent", fontsize=9)
    fig.legend(handles=[Line2D([], [], color=INK, lw=1.6, label="cell mean")],
               loc="lower center", fontsize=8.5, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout()
    return fig


if RATINGS is not None:
    fig = fig_spread(RATINGS, list(LISTEN_WIDE))
    save(fig, "ch2_listening_spread", subdir="listening")

## 7. Fine-tuning loss curves (asset A0)

The per-accent LoRA fine-tunes, read straight from the TensorBoard event files
the fork's trainer writes (`runs/lora_<accent>/events.out.tfevents.*`). They are
gitignored, so pull them down first:

```
rsync -av --include='*/' --include='events.out.tfevents.*' --exclude='*' \
    <user>@eddie.ecdf.ed.ac.uk:/exports/.../slp-diss/AccentVector/runs/ runs/
```

No tensorboard/tbparse dependency: the event files are parsed below directly
(TFRecord frames plus the handful of protobuf fields a scalar summary uses).

In [ ]:
import struct


def _tfrecords(path):
    """TFRecord frames: u64 length | u32 crc | payload | u32 crc.

    CRCs are not checked (nothing here is corruption-sensitive); a short read just
    ends the file, which is what a job killed mid-write looks like.
    """
    with open(path, "rb") as fh:
        while True:
            head = fh.read(12)
            if len(head) < 12:
                return
            (length,) = struct.unpack("<Q", head[:8])
            payload = fh.read(length)
            if len(payload) < length or len(fh.read(4)) < 4:
                return
            yield payload


def _varint(buf, i):
    val = shift = 0
    while True:
        b = buf[i]
        i += 1
        val |= (b & 0x7F) << shift
        if not b & 0x80:
            return val, i
        shift += 7


def _fields(buf):
    """(field_number, wire_type, payload) for one protobuf message."""
    i, n = 0, len(buf)
    while i < n:
        key, i = _varint(buf, i)
        fno, wt = key >> 3, key & 7
        if wt == 0:
            val, i = _varint(buf, i)
        elif wt == 1:
            val, i = buf[i:i + 8], i + 8
        elif wt == 2:
            ln, i = _varint(buf, i)
            val, i = buf[i:i + ln], i + ln
        elif wt == 5:
            val, i = buf[i:i + 4], i + 4
        else:                                   # groups: not used by Event/Summary
            return
        yield fno, wt, val


def read_event_file(path):
    """[(wall_time, step, tag, value)] for every simple_value scalar in one file.

    Event{1: wall_time, 2: step, 5: Summary}, Summary{1: Value},
    Summary.Value{1: tag, 2: simple_value}.
    """
    rows = []
    for rec in _tfrecords(path):
        wall, step, summaries = float("nan"), 0, []
        for fno, wt, val in _fields(rec):
            if fno == 1 and wt == 1:
                wall = struct.unpack("<d", val)[0]
            elif fno == 2 and wt == 0:
                step = val
            elif fno == 5 and wt == 2:
                summaries.append(val)
        for summ in summaries:
            for fno, wt, val in _fields(summ):
                if fno != 1 or wt != 2:
                    continue
                tag = value = None
                for f2, w2, v2 in _fields(val):
                    if f2 == 1 and w2 == 2:
                        tag = v2.decode("utf-8", "replace")
                    elif f2 == 2 and w2 == 5:
                        value = struct.unpack("<f", v2)[0]
                if tag is not None and value is not None:
                    rows.append((wall, int(step), tag, float(value)))
    return rows


def find_runs(runs_root, accents):
    """{accent: [event files]} — any event file whose path mentions the accent, so
    lora_<accent>/, lora_<accent>_resume/ and <accent>/run2/ all get picked up."""
    runs_root = Path(runs_root)
    if not runs_root.is_dir():
        return {}
    out = {a: [] for a in accents}
    for f in sorted(runs_root.rglob("events.out.tfevents.*")):
        rel = str(f.relative_to(runs_root)).lower()
        for a in accents:
            if a in rel:
                out[a].append(f)
                break
    return {a: fs for a, fs in out.items() if fs}


def load_run(files):
    """Merged (step, tag, value) for one accent. Chained/resumed jobs repeat steps:
    keep the latest write per (tag, step)."""
    rows = []
    for f in sorted(files, key=lambda p: p.name):   # filename embeds the start time
        rows += read_event_file(f)
    if not rows:
        return pd.DataFrame(columns=["wall_time", "step", "tag", "value"])
    df = pd.DataFrame(rows, columns=["wall_time", "step", "tag", "value"])
    df = df.sort_values(["tag", "step", "wall_time"], kind="stable")
    df = df.drop_duplicates(subset=["tag", "step"], keep="last")
    return df[np.isfinite(df.value)].reset_index(drop=True)


def series(df, tag):
    d = df[df.tag == tag].sort_values("step")
    return d.step.to_numpy(float), d.value.to_numpy(float)


def ema(y, weight):
    """TensorBoard's smoothing: exponential moving average with bias correction."""
    out = np.empty_like(y, dtype=float)
    last = num = 0.0
    for i, v in enumerate(y):
        last = last * weight + (1 - weight) * v
        num = num * weight + (1 - weight)
        out[i] = last / num if num else v
    return out


SMOOTH = 0.95
FOUND = find_runs(RUNS_ROOT, ACCENTS)
RUNS = {}
for acc in ACCENTS:
    if acc not in FOUND:
        continue
    df = load_run(FOUND[acc])
    if df.empty:
        continue
    RUNS[acc] = df
    n = df[df.tag == "loss"].step.max()
    print(f"{acc:>9}: {len(FOUND[acc])} file(s), tags={sorted(df.tag.unique())}, "
          f"{int(n) if pd.notna(n) else 0:,} updates")
if not RUNS:
    print(f"no event files under {RUNS_ROOT} — rsync runs/ down from Eddie to draw §7")

In [ ]:
def fig_train_loss(runs, smooth=SMOOTH, ylim=None):
    """All accents on one axes: raw per-update loss faint, EMA on top."""
    fig, ax = plt.subplots(figsize=(7.0, 4.2))
    used = {}
    for acc, df in runs.items():
        x, y = series(df, "loss")
        if not len(x):
            continue
        st = style(acc)
        ax.plot(x, y, color=st["c"], lw=0.5, alpha=0.15, zorder=1)
        ax.plot(x, ema(y, smooth), color=st["c"], ls=st["ls"], lw=1.8, zorder=2)
        used[acc] = int(x.max())
    ax.set_xlabel("fine-tuning update")
    ax.set_ylabel("training loss")
    ax.set_title(f"LoRA fine-tuning loss per accent (EMA, α={smooth:g}; raw faint)",
                 fontsize=10)
    if ylim:
        ax.set_ylim(*ylim)
    ax.legend(handles=[Line2D([], [], color=style(a)["c"], ls=style(a)["ls"], lw=1.8,
                              label=f"{style(a)['label']} ({n:,} updates)")
                       for a, n in used.items()], fontsize=8, loc="upper right")
    return fig


def fig_loss_panels(runs, smooth=SMOOTH, ncols=3):
    """One panel per accent: smoothed train loss + validation loss."""
    accs = list(runs)
    fig, axes = panel_axes(int(np.ceil(len(accs) / ncols)), ncols, w=3.5, h=2.9)
    for ax, acc in zip(axes, accs):
        st, df = style(acc), runs[acc]
        x, y = series(df, "loss")
        if len(x):
            ax.plot(x, y, color=st["c"], lw=0.5, alpha=0.15)
            ax.plot(x, ema(y, smooth), color=st["c"], ls="-", lw=1.6)
        vx, vy = series(df, "valid_loss")
        if len(vx):
            ax.plot(vx, vy, color=INK, ls="--", lw=1.0, marker=st["m"], ms=3)
        ax.set_title(st["label"], fontsize=10)
        ax.set_xlabel("update", fontsize=9)
        ax.tick_params(labelsize=8)
    axes[0].set_ylabel("loss", fontsize=9)
    for ax in axes[len(accs):]:
        ax.axis("off")
    fig.legend(handles=[Line2D([], [], color=MUTED, lw=1.6, label="train (EMA)"),
                        Line2D([], [], color=INK, ls="--", lw=1.0, marker="o", ms=3,
                               label="validation")],
               loc="lower center", ncol=2, fontsize=8.5, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout()
    return fig


def loss_summary(runs, smooth=SMOOTH):
    rows = []
    for acc, df in runs.items():
        x, y = series(df, "loss")
        vx, vy = series(df, "valid_loss")
        sm = ema(y, smooth) if len(y) else np.array([])
        rows.append({"accent": acc, "updates": int(x.max()) if len(x) else 0,
                     "train_loss_final_ema": float(sm[-1]) if len(sm) else np.nan,
                     "train_loss_min_ema": float(sm.min()) if len(sm) else np.nan,
                     "valid_loss_final": float(vy[-1]) if len(vy) else np.nan,
                     "valid_loss_best": float(vy.min()) if len(vy) else np.nan,
                     "valid_best_update": int(vx[int(vy.argmin())]) if len(vy) else -1,
                     "n_valid_points": int(len(vy))})
    return pd.DataFrame(rows)


if RUNS:
    save(fig_train_loss(RUNS), "fig_train_loss", subdir="training")
    save(fig_loss_panels(RUNS), "fig_loss_panels", subdir="training")
    LOSS_SUMMARY = loss_summary(RUNS)
    if SAVE:
        (FIGDIR / "training").mkdir(parents=True, exist_ok=True)
        LOSS_SUMMARY.to_csv(FIGDIR / "training" / "loss_summary.csv", index=False)
    print(LOSS_SUMMARY.set_index("accent").round(4).to_string())

## 8. Diagnostics and appendix figures

Not chapter figures. These are the views to reach for when a headline figure
looks wrong, and the evidence behind two decisions the chapters state without
re-deriving. Each is a function plus a call you can point at any accent.

### 8.1 Leakage onset, with censoring made explicit

Where an onset is absent the signal *never* crossed threshold in the swept α
range. That is a censored observation, not missing data: drawn at the top of the
axis with an open marker rather than silently dropped.

In [ ]:
def fig_onsets(footer, long_df, accent):
    keys = [("wer_leak_onset", "WER leakage-onset α  ↑", "WER never reached threshold"),
            ("lid_leak_onset", "LID leakage-onset α  ↑", "P(English) never fell below 0.5")]
    max_a = float(long_df.alpha.max())
    fig, axes = panel_axes(1, 2, w=4.4, h=3.2)
    for ax, (key, lab, note) in zip(axes, keys):
        all_steps = sorted(footer.step.unique())
        for ref, rs in REF_STYLE.items():
            for spk, ss in SPEAKER_STYLE.items():
                f = footer[(footer.ref_kind == ref) & (footer.speaker == spk)]
                if f.empty:
                    continue
                f = f.set_index("step").reindex(all_steps)
                y = f[key] if key in f.columns else pd.Series(np.nan, index=all_steps)
                obs = y.dropna()
                if len(obs):
                    ax.plot(obs.index, obs.values, ss["ls"], color=rs["c"],
                            marker=ss["m"], ms=3.6, lw=1.4, mew=0)
                cens = [s for s in all_steps if pd.isna(y.get(s, np.nan))]
                if cens:
                    ax.plot(cens, [max_a * 1.06] * len(cens), ss["m"], mfc="none",
                            mec=rs["c"], mew=1.1, ms=5, ls="none")
        ax.axhline(max_a, color=GRID, lw=0.8, ls=":")
        ax.text(0.01, max_a * 1.09, f"censored — {note}", fontsize=7, color=MUTED,
                transform=ax.get_yaxis_transform(), va="bottom")
        ax.set_title(lab, fontsize=10)
        ax.set_xlabel("training step")
        ax.set_ylabel("α at which content is treated as leaked")
        ax.set_ylim(-0.03, max_a * 1.22)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
    handles = [Line2D([], [], color=r["c"], lw=2.2, label=REF_LABEL[k])
               for k, r in REF_STYLE.items()]
    handles += [Line2D([], [], color=MUTED, lw=1.4, ls=s["ls"], marker=s["m"], ms=3.4,
                       label=s["label"]) for s in SPEAKER_STYLE.values()]
    handles.append(Line2D([], [], ls="none", marker="o", mfc="none", mec=MUTED, ms=5,
                          label="censored (never leaked)"))
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=8,
               bbox_to_anchor=(0.5, -0.08))
    fig.suptitle(f"{style(accent)['label']} — leakage onset vs training step",
                 fontsize=11, y=1.02)
    fig.tight_layout()
    return fig


DIAG_ACCENT = list(DATA)[0]
fig = fig_onsets(FOOT[DIAG_ACCENT], DATA[DIAG_ACCENT], DIAG_ACCENT)
save(fig, f"fig_onsets_{DIAG_ACCENT}", subdir="diagnostics")

### 8.2 Nothing pooled

The §2 panels pool the two prompt speakers. This is the unpooled view — one line
per (ref_kind × speaker), four lines, never averaged — for checking that a
pooled trend is not one speaker's artefact.

In [ ]:
def fig_unpooled(long_df, accent, step):
    d = long_df[long_df.step == step]
    cols = [p for p in PANELS if p[0] in d.columns and d[p[0]].notna().any()]
    fig, axes = panel_axes(1, len(cols), w=2.6, h=2.6)
    for ax, (col, title, ylab, arrow, scale) in zip(axes, cols):
        for ref, rs in REF_STYLE.items():
            for spk, ss in SPEAKER_STYLE.items():
                s = d[(d.ref_kind == ref) & (d.speaker == spk)].dropna(subset=[col])
                if s.empty:
                    continue
                s = s.sort_values("alpha")
                ax.plot(s.alpha, s[col] * scale, ss["ls"], color=rs["c"], marker=ss["m"],
                        ms=3.4, lw=1.4, mew=0, alpha=0.95)
        ax.set_xlabel("α")
        finish_panel(ax, title, ylab, arrow)
    handles = [Line2D([], [], color=r["c"], lw=2.2, label=REF_LABEL[k])
               for k, r in REF_STYLE.items()]
    handles += [Line2D([], [], color=MUTED, lw=1.4, ls=s["ls"], marker=s["m"], ms=3.4,
                       label=s["label"]) for s in SPEAKER_STYLE.values()]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8,
               bbox_to_anchor=(0.5, -0.10))
    fig.suptitle(f"{style(accent)['label']} — every metric, every speaker, no pooling "
                 f"(step {step:,})", fontsize=11, y=1.04)
    fig.tight_layout()
    return fig


fig = fig_unpooled(DATA[DIAG_ACCENT], DIAG_ACCENT, FINALS[DIAG_ACCENT])
save(fig, f"fig_unpooled_{DIAG_ACCENT}", subdir="diagnostics")

### 8.3 Any metric across every checkpoint × α

One generic heatmap replacing the three near-identical cube views the old detail
module carried (`fig_rq1_by_step`, `fig_rq2_detail`, `fig_rq3_seg_by_step`): give
it any column and it draws the full step × α field for both prompts, so a trend
and a speckle of evaluation noise can be told apart at a glance. Signed
quantities get the diverging ramp pinned with neutral grey exactly on zero;
unsigned ones get the sequential ramp.

In [ ]:
def cell_matrix(sub, col, steps, alphas):
    """step × α matrix of `col`; speakers averaged, since a cell is one number."""
    M = np.full((len(steps), len(alphas)), np.nan)
    piv = sub.groupby(["step", "alpha"])[col].mean()
    for i, s in enumerate(steps):
        for j, a in enumerate(alphas):
            if (s, a) in piv.index:
                M[i, j] = piv.loc[(s, a)]
    return M


def fig_metric_heat(long_df, accent, cols=None, signed=None):
    """step × α heatmap per metric, one row per metric, L1 and GAE side by side."""
    cols = cols or [p[0] for p in PANELS]
    cols = [(c, dict((p[0], p[1]) for p in PANELS).get(c, c))
            for c in cols if c in long_df.columns and long_df[c].notna().any()]
    signed = signed or {"seg_closure", "supra_closure_mean"}
    steps = sorted(long_df.step.unique())
    alphas = sorted(long_df.alpha.unique())
    if len(steps) < 3:
        print(f"{accent}: <3 checkpoints; skipping heatmap")
        return None
    fig, axes = plt.subplots(len(cols), 2, figsize=(3.4 * 2, 1.9 * len(cols)),
                             squeeze=False, layout="constrained")
    for r, (col, lab) in enumerate(cols):
        vals = long_df[col].dropna()
        vmin, vmax = float(vals.min()), float(vals.max())
        # one shared colour scale per metric row, so the two prompts are comparable
        kw = ({"cmap": DIV, "norm": TwoSlopeNorm(vmin=min(vmin, -1e-6), vcenter=0.0,
                                                 vmax=max(vmax, 1e-6))}
              if col in signed else {"cmap": SEQ, "vmin": vmin, "vmax": vmax})
        for c, ref in enumerate(("l1", "native")):
            ax = axes[r][c]
            im = ax.imshow(cell_matrix(long_df[long_df.ref_kind == ref], col, steps, alphas),
                           aspect="auto", origin="lower", **kw)
            ax.set_xticks(range(len(alphas)),
                          [f"{a:g}" for a in alphas] if r == len(cols) - 1 else [], fontsize=7)
            ax.set_yticks(range(len(steps)),
                          [f"{s // 1000}k" for s in steps] if c == 0 else [], fontsize=7)
            ax.grid(False)
            ax.set_title(f"{lab} — {REF_LABEL[ref]}", fontsize=8)
            if r == len(cols) - 1:
                ax.set_xlabel("α")
            if c == 0:
                ax.set_ylabel("step", fontsize=8)
        cb = fig.colorbar(im, ax=list(axes[r]), fraction=0.035, pad=0.015)
        cb.ax.tick_params(labelsize=7)
        cb.outline.set_visible(False)
    fig.suptitle(f"{style(accent)['label']} — every metric at every checkpoint × α "
                 f"(speakers averaged per cell)", fontsize=11)
    return fig


fig = fig_metric_heat(DATA[DIAG_ACCENT], DIAG_ACCENT)
if fig is not None:
    save(fig, f"fig_metric_heat_{DIAG_ACCENT}", subdir="diagnostics")

### 8.4 Why the old RQ3 composite was retired

The one place `supra_closure_mean` still appears. Each per-feature gap closure
on a symlog axis, with the composite that averages them boxed. A closure is
$(x_\alpha - x_0)/(x_{\text{nat}} - x_0)$, so a feature whose natural target sits
almost on its own baseline has a near-zero denominator and a closure that blows
up — and the composite, being an unweighted mean, goes with it. That is the
figure behind §5's replacement scale; the chapters cite it and use nothing else
from the old scale.

In [ ]:
def fig_closure_diagnostic(long_df, accent, step):
    d = long_df[long_df.step == step]
    cols = ([("seg_closure", "Segmental closure (PPG-KL)")]
            + [(f"{f}_closure", SUPRA_LABEL[f]) for f in SUPRA]
            + [("supra_closure_mean", "supra_closure_mean (the composite)")])
    cols = [c for c in cols if c[0] in d.columns]
    fig, axes = panel_axes(2, 4, w=2.6, h=2.4)
    for ax, (col, lab) in zip(axes, cols):
        for ref, rs in REF_STYLE.items():
            for spk, ss in SPEAKER_STYLE.items():
                s = d[(d.ref_kind == ref) & (d.speaker == spk)].dropna(subset=[col])
                if s.empty:
                    continue
                s = s.sort_values("alpha")
                ax.plot(s.alpha, s[col], ss["ls"], color=rs["c"], marker=ss["m"],
                        ms=3.4, lw=1.4, mew=0)
        ax.axhline(0, color=GRID, lw=0.8, ls=":")                       # no movement
        ax.axhline(1, color="#0ca30c", lw=0.8, ls="--", alpha=0.7)      # fully closed
        ax.set_yscale("symlog", linthresh=1.0, linscale=0.9)
        ax.set_title(lab, fontsize=8)
        ax.set_xlabel("α")
        ax.margins(x=0.04)
        if col == "supra_closure_mean":
            for s_ in ax.spines.values():
                s_.set_edgecolor(INK)
                s_.set_linewidth(1.2)
        peak = d[col].abs().max()
        if np.isfinite(peak) and peak > 5:
            ax.annotate(f"peak |closure| = {peak:.1f}", (0.03, 0.92), fontsize=7,
                        color="#B4451F", xycoords="axes fraction", va="top")
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    axes[0].set_ylabel("gap closure  (0 = no move, 1 = reached natural)", fontsize=7.5)
    fig.suptitle(f"{style(accent)['label']} — retired RQ3 composite (symlog). The boxed "
                 f"panel is the mean of the six before it.", fontsize=11, y=1.02)
    fig.tight_layout()
    return fig


for acc in DATA:
    fig = fig_closure_diagnostic(DATA[acc], acc, FINALS[acc])
    save(fig, f"fig_rq3_closure_retired_{acc}", subdir="diagnostics")
    break      # one accent is enough to make the point; loop over DATA for all

### 8.5 Tidy dump

Every number behind every figure above, in long form, so any value in any panel
can be traced back — plus the recovered natural targets, which are derived
rather than stored.

In [ ]:
def write_long(data3, natural, out_csv):
    id_cols = ["ref_kind", "speaker", "step", "alpha"]
    frames = []
    for acc, df in data3.items():
        metrics = [c for c in df.columns if c not in id_cols + ["n"]]
        long = df.melt(id_vars=id_cols, value_vars=metrics,
                       var_name="metric", value_name="value").dropna(subset=["value"])
        long.insert(0, "accent", acc)
        frames.append(long)
    long = pd.concat(frames, ignore_index=True).sort_values(["accent"] + id_cols + ["metric"])
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    long.to_csv(out_csv, index=False)

    nat_rows = [dict(accent=acc, ref_kind=r, speaker=s, step=st, feature=f, x_natural=v)
                for acc, nat in natural.items() for (r, s, st, f), v in sorted(nat.items())]
    nat_csv = Path(out_csv).with_name("natural_targets.csv")
    pd.DataFrame(nat_rows).to_csv(nat_csv, index=False)
    print(f"wrote {out_csv} ({len(long):,} rows) and {nat_csv}")


if SAVE:
    write_long(DATA3, NATURAL, FIGDIR / "diagnostics" / "detail_long.csv")